# Adaptation ladder for dental foundation models under a safety envelope (DENTEX to ADCD)

This notebook runs every experiment of the paper, top to bottom, and writes all results to one folder.

**Stages**

| Section | Stage | What it does |
|---|---|---|
| 3, 6 | A | Parses DENTEX teeth and per-box ADCD lesions, builds tooth-context crops, computes and caches embeddings for the three backbones |
| 7b | checks | Embedding-cache integrity and duplicate-radiograph checks |
| 8 | B | Frozen-backbone ladder: label-free recalibration (Rung 0), head re-fit (Rung 1), embedding-space corrections |
| 8b | B2 | Two-backbone fusion |
| 9 | C | Live-trunk parameter-efficient fine-tuning (Rung 2) and full-trunk fine-tuning with forgetting controls (Rung 3) |
| 10 | D | Two-center training at matched label budgets, condition-wise transferability test, box-geometry baseline |
| 10b | G | Validity check for a healthy-tooth analysis on DENTEX |
| 11 | E | Confidence intervals, paired differences, source preservation, AUROC-coverage, gate refusal matrix |
| 12, 13 | | Completeness check and results archive |

**Before running**

1. Download the data (see `docs/data.md`) and set `DENTEX_DIR` and `ADCD_DIR`, either as environment variables or in section 2.
2. DINOv3 is a gated model on Hugging Face: accept its licence on the model page and run `huggingface-cli login` once.
3. Install the dependencies with `pip install -r requirements.txt`.

Stage C saves after every run and skips runs that are already complete when the notebook is restarted. On an Apple M3 Ultra the whole notebook took about 8 hours, most of it in Stage C.

`SRC_TOL` (source-preservation tolerance) and `HARMONIZE_CARIES` are set in section 2. The values used in the paper are 0.02 and `True`.


## 1. Setup

In [ ]:
import sys, os, json, math, glob, random, time, copy, csv
# Dependencies: pip install -r requirements.txt  (see the repository README)
import numpy as np, torch
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| device:", DEVICE)
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)


## 2. Paths and configuration
Set the two dataset folders (or the environment variables `DENTEX_DIR` and `ADCD_DIR`). `SRC_TOL` is the source-preservation tolerance and must be fixed before any result is examined.


In [ ]:
# ---- PATHS ----
DENTEX_DIR   = os.environ.get("DENTEX_DIR", "data/DENTEX/training_data/quadrant-enumeration-disease")
DENTEX_TRAIN = os.path.dirname(DENTEX_DIR.rstrip("/"))   # holds quadrant_enumeration/ for Stage G
IMG_DIR      = os.path.join(DENTEX_DIR, "xrays")
ADCD_DIR     = os.environ.get("ADCD_DIR", "data/ADCD/panoramic_radiography_yolo_dataset_14_classes")
# ADCD layout: train/, valid/, test/ (pooled and re-split by radiograph) and test_alte_cabinete/Ext-validation/
# (external validation set from other centers, held out entirely and scored as a second target).
OUT2  = os.environ.get("RESULTS_DIR", "outputs"); os.makedirs(OUT2, exist_ok=True)
CACHE = os.path.join(OUT2, "emb_cache"); os.makedirs(CACHE, exist_ok=True)
assert os.path.isdir(DENTEX_DIR), f"DENTEX_DIR not found: {DENTEX_DIR}"
assert os.path.isdir(ADCD_DIR), f"ADCD_DIR not found: {ADCD_DIR}  <-- set it in this cell"

# ---- CONFIG ----
CLASSES        = ["Impacted", "Caries", "Periapical Lesion", "Deep Caries"]
TARGET_CLASSES = ["Impacted", "Caries", "Periapical Lesion"]          # shared with ADCD
NUM_DX = len(CLASSES); CLASS_IDX = {c: i for i, c in enumerate(CLASSES)}
ADCD_CLASS_TO_DX = {4: "Caries", 6: "Impacted", 7: "Periapical Lesion"}   # verified against data.yaml in Stage A
DELTA, MAHA_RIDGE, OOD_Q = 1.5, 1e-2, 0.95
SEEDS = (42, 43, 44)
PAD   = 0.15                                                            # crop margin, both datasets
SRC_TOL = 0.02     # prespecified: maximum acceptable drop in DENTEX macro AUROC
# Caries harmonization: ADCD does not separate caries by depth, so DENTEX
# "Deep Caries" teeth are also labeled "Caries" for every cross-dataset analysis. The separate
# Deep Caries column is kept for the 4-condition source metrics. Run once with True (primary);
# optionally run again with False as a sensitivity check (Stages A, B, B2, D, E; Stage C not needed).
HARMONIZE_CARIES = True
ADCD_CROP_MODE = "tooth_context"   # output namespace reflects the exact crop protocol
V2_NAME = f"v5_{'harmonized' if HARMONIZE_CARIES else 'separate'}_{ADCD_CROP_MODE}"
print("OUT2:", OUT2, "| results folder:", V2_NAME)

## 2b. Results folder


In [ ]:
os.makedirs(os.path.join(OUT2, V2_NAME), exist_ok=True)
print("results folder:", os.path.join(OUT2, V2_NAME))


## 3. DENTEX parse and crops

In [ ]:
from PIL import Image
if not os.path.isdir(IMG_DIR):
    IMG_DIR = None
    for root, _, files in os.walk(DENTEX_DIR):
        if "_results" in root: continue
        if any(f.lower().endswith((".png", ".jpg")) for f in files): IMG_DIR = root; break
assert IMG_DIR and os.path.isdir(IMG_DIR), "DENTEX image folder not found"
ANN_PATH = None
for root, _, files in os.walk(DENTEX_DIR):
    if "_results" in root: continue
    for f in files:
        if f.endswith(".json"):
            try: j = json.load(open(os.path.join(root, f)))
            except Exception: continue
            if isinstance(j, dict) and "annotations" in j and any("category_id_3" in a for a in j.get("annotations", [])):
                ANN_PATH = os.path.join(root, f); break
    if ANN_PATH: break
assert ANN_PATH, "DENTEX diagnosis JSON not found"
coco = json.load(open(ANN_PATH))
name_by_id = {c["id"]: c["name"] for c in coco["categories_3"]}
img_file = {im["id"]: im["file_name"] for im in coco["images"]}
# one multi-label tooth instance per (image, quadrant, tooth number); box = union of its diagnosis boxes
tooth = {}
for a in coco["annotations"]:
    key = (a["image_id"], a.get("category_id_1", -1), a.get("category_id_2", -1))
    if key not in tooth:
        tooth[key] = {"image_id": a["image_id"], "file": img_file[a["image_id"]], "bbox": list(a["bbox"]), "y": np.zeros(NUM_DX, np.float32)}
    nm = name_by_id.get(a["category_id_3"])
    if nm in CLASS_IDX: tooth[key]["y"][CLASS_IDX[nm]] = 1.0
    bx, by, bw, bh = tooth[key]["bbox"]; ax, ay, aw, ah = a["bbox"]
    x0, y0 = min(bx, ax), min(by, ay); x1, y1 = max(bx+bw, ax+aw), max(by+bh, ay+ah)
    tooth[key]["bbox"] = [x0, y0, x1-x0, y1-y0]
teeth = list(tooth.values()); Y = np.stack([t["y"] for t in teeth])
if HARMONIZE_CARIES:
    _ci, _di = CLASS_IDX["Caries"], CLASS_IDX["Deep Caries"]
    _added = int(((Y[:, _di] > 0) & (Y[:, _ci] == 0)).sum())
    Y[:, _ci] = np.maximum(Y[:, _ci], Y[:, _di])
    for t, yy in zip(teeth, Y): t["y"] = yy.copy()
    print(f"caries harmonized: {_added} deep-caries-only teeth now also labeled Caries")
print(f"{len(teeth)} DENTEX teeth from {len(img_file)} images | positives {dict(zip(CLASSES, Y.sum(0).astype(int).tolist()))}")

def grouped_split(teeth, seed, vf=0.15, tf=0.15):
    ids = sorted({t["image_id"] for t in teeth}); rng = np.random.RandomState(seed); rng.shuffle(ids)
    n = len(ids); nte = int(n*tf); nva = int(n*vf); te = set(ids[:nte]); va = set(ids[nte:nte+nva])
    tr_, va_, te_ = [], [], []
    for i, t in enumerate(teeth):
        (te_ if t["image_id"] in te else va_ if t["image_id"] in va else tr_).append(i)
    return tr_, va_, te_

_imc = {}
def load_image(f):
    if f not in _imc:
        if len(_imc) > 128: _imc.clear()
        p = os.path.join(IMG_DIR, os.path.basename(f))
        if not os.path.exists(p):
            hits = glob.glob(os.path.join(DENTEX_DIR, "**", os.path.basename(f)), recursive=True)
            p = hits[0] if hits else p
        _imc[f] = Image.open(p).convert("RGB")
    return _imc[f]

def crop_tooth(img, bbox, pad=PAD):
    W, H = img.size; x, y, w, h = bbox; px, py = w*pad, h*pad
    x0 = max(0, int(x-px)); y0 = max(0, int(y-py)); x1 = min(W, int(x+w+px)); y1 = min(H, int(y+h+py))
    return img if (x1 <= x0 or y1 <= y0) else img.crop((x0, y0, x1, y1))

def get_crop(i):
    t = teeth[i]; return crop_tooth(load_image(t["file"]), t["bbox"])
print("sample crop size:", get_crop(0).size)

## 4. Backbones
Dental ViT-L input size is read from the checkpoint. DINOv3 needs the gated weights in your local Hugging Face cache.

In [ ]:
import torchvision.transforms as T

def load_biomedclip():
    import open_clip
    model, _, preprocess = open_clip.create_model_and_transforms("hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")
    return model.visual.to(DEVICE).eval(), preprocess

def load_dental_vitl_frozen():
    from huggingface_hub import hf_hub_download
    if not os.path.isdir("ViT-pytorch"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jeonsworld/ViT-pytorch"], check=True)
    if "ViT-pytorch" not in sys.path: sys.path.insert(0, "ViT-pytorch")
    from models.modeling import VisionTransformer, CONFIGS
    ck = hf_hub_download(repo_id="Bryceee/Teeth_Visual_Experts_Models",
                         filename="Teeth_Visual_Experts_ViT_L_periapical_images_cls_7diseases.bin")
    state = torch.load(ck, map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state: state = state["state_dict"]
    pos = state["transformer.embeddings.position_embeddings"].shape[1]
    img = int(round(math.sqrt(pos - 1))) * 16; nH = state["head.weight"].shape[0]
    m = VisionTransformer(CONFIGS["ViT-L_16"], img_size=img, num_classes=nH, zero_head=False)
    missing, unexpected = m.load_state_dict(state, strict=False)
    print(f"Dental ViT-L: input {img}px, {nH}-class head, missing {len(missing)}, unexpected {len(unexpected)}")
    pre = T.Compose([T.Resize((img, img)), T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)])
    class DentalEmbed(torch.nn.Module):
        def __init__(self, vit): super().__init__(); self.vit = vit
        @torch.no_grad()
        def forward(self, x): seq, _ = self.vit.transformer(x); return seq[:, 0]
    return DentalEmbed(m).to(DEVICE).eval(), pre, img

def load_dinov3():
    from transformers import AutoImageProcessor, AutoModel
    DID = "facebook/dinov3-vitl16-pretrain-lvd1689m"
    proc = AutoImageProcessor.from_pretrained(DID); model = AutoModel.from_pretrained(DID).to(DEVICE).eval()
    return model, proc

@torch.no_grad()
def _embed_torch(model, preprocess, pils, bs=32, l2norm=True):
    outs = []
    for k in range(0, len(pils), bs):
        x = torch.stack([preprocess(p) for p in pils[k:k+bs]]).to(DEVICE)
        f = model(x); f = f[0] if isinstance(f, (tuple, list)) else f
        f = f.float().cpu()
        if l2norm: f = torch.nn.functional.normalize(f, dim=1)
        outs.append(f.numpy())
    return np.concatenate(outs, 0).astype(np.float32)

@torch.no_grad()
def _embed_dinov3(model, proc, pils, bs=16):
    outs = []
    for k in range(0, len(pils), bs):
        inp = proc(images=[p.convert("RGB") for p in pils[k:k+bs]], return_tensors="pt").to(DEVICE)
        o = model(**inp); f = getattr(o, "pooler_output", None)
        if f is None: f = o.last_hidden_state[:, 0]
        outs.append(torch.nn.functional.normalize(f.float().cpu(), dim=1).numpy())
    return np.concatenate(outs, 0).astype(np.float32)

_EMBEDDERS = {}
def make_embedder(name):
    if name not in _EMBEDDERS:
        if name == "biomedclip":
            m, pre = load_biomedclip(); _EMBEDDERS[name] = lambda pils: _embed_torch(m, pre, pils)
        elif name == "dental_vitl":
            m, pre, img = load_dental_vitl_frozen(); _EMBEDDERS[name] = lambda pils: _embed_torch(m, pre, pils)
            os.makedirs(os.path.join(OUT2, V2_NAME), exist_ok=True)
            json.dump({"dental_vitl_input_px": img}, open(os.path.join(OUT2, V2_NAME, "dental_vitl_config.json"), "w"))
        elif name == "dinov3":
            m, proc = load_dinov3(); _EMBEDDERS[name] = lambda pils: _embed_dinov3(m, proc, pils)
        else:
            raise ValueError(name)
    return _EMBEDDERS[name]
print("backbone loaders ready")

## 5. Safety envelope
Calibrated probes, clipped residual head (bound = DELTA), Mahalanobis gate (ridge 1e-2, tau = 95th percentile on validation).

In [ ]:
import torch.nn as nn
from sklearn.linear_model import LogisticRegression

def fit_baseline(Xtr,Ytr,k):
    return [LogisticRegression(max_iter=1000,C=1.0).fit(Xtr,Ytr[:,c]) if len(np.unique(Ytr[:,c]))>=2 else None
            for c in range(k)]
def baseline_logits(probes,X,k):
    out=np.zeros((len(X),k),np.float32)
    for c,p in enumerate(probes): out[:,c]=p.decision_function(X) if p is not None else 0.0
    return out
def fit_temperature(logits,y):
    Tt=nn.Parameter(torch.ones(1)); z=torch.tensor(logits); yt=torch.tensor(y)
    opt=torch.optim.LBFGS([Tt],lr=0.1,max_iter=60); bce=nn.BCEWithLogitsLoss()
    def closure(): opt.zero_grad(); l=bce(z/Tt.clamp(min=0.05),yt); l.backward(); return l
    opt.step(closure); return float(Tt.detach().clamp(min=0.05))
def ece_score(y,p,nb=10):
    b=np.linspace(0,1,nb+1); e=0.0
    for lo,hi in zip(b[:-1],b[1:]):
        m=(p>lo)&(p<=hi)
        if m.sum()==0: continue
        e+=(m.sum()/len(p))*abs(p[m].mean()-y[m].mean())
    return float(e)
class EnvelopeHead(nn.Module):
    def __init__(self,d,k,delta):
        super().__init__(); self.delta=delta
        self.net=nn.Sequential(nn.Linear(d,256),nn.ReLU(),nn.Dropout(0.2),nn.Linear(256,k))
    def residual(self,x): return torch.clamp(self.net(x),-self.delta,self.delta)
    def forward(self,x,base): return base+self.residual(x)
def train_envelope_head(X,base,Yt,Xva,bva,Yva,d,k,epochs=60,lr=1e-3):
    from torch.utils.data import TensorDataset,DataLoader
    head=EnvelopeHead(d,k,DELTA).to(DEVICE); opt=torch.optim.AdamW(head.parameters(),lr=lr,weight_decay=1e-4)
    crit=nn.BCEWithLogitsLoss()
    dl=DataLoader(TensorDataset(torch.tensor(X),torch.tensor(base),torch.tensor(Yt)),batch_size=64,shuffle=True)
    best,bs=1e9,None
    for _ in range(epochs):
        head.train()
        for xb,bb,yb in dl:
            xb,bb,yb=xb.to(DEVICE),bb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad(); loss=crit(head(xb,bb),yb); loss.backward(); opt.step()
        head.eval()
        with torch.no_grad():
            vl=crit(head(torch.tensor(Xva).to(DEVICE),torch.tensor(bva).to(DEVICE)),torch.tensor(Yva).to(DEVICE)).item()
        if vl<best: best,bs=vl,{k_:v.detach().cpu().clone() for k_,v in head.state_dict().items()}
    head.load_state_dict(bs); return head
def maha_scorer(Xtr,ridge=MAHA_RIDGE):
    mu=Xtr.mean(0); cov=np.cov(Xtr.T)+np.eye(Xtr.shape[1])*ridge; ci=np.linalg.inv(cov)
    return (lambda X: np.einsum("ni,ij,nj->n",X-mu,ci,X-mu)), mu, ci
class Envelope:
    def fit(self,Xtr,Ytr,Xva,Yva,epochs=60):
        self.probes=fit_baseline(Xtr,Ytr,NUM_DX)
        ztr=baseline_logits(self.probes,Xtr,NUM_DX); zva=baseline_logits(self.probes,Xva,NUM_DX)
        self.T=fit_temperature(zva,Yva)
        self.head=train_envelope_head(Xtr,ztr/self.T,Ytr,Xva,zva/self.T,Yva,Xtr.shape[1],NUM_DX,epochs)
        self._score,self.mu,self.ci=maha_scorer(Xtr); self.tau=float(np.quantile(self._score(Xva),OOD_Q)); return self
    def proba(self,X):
        b=baseline_logits(self.probes,X,NUM_DX)/self.T; self.head.eval()
        with torch.no_grad(): f=self.head(torch.tensor(X).to(DEVICE),torch.tensor(b).to(DEVICE)).cpu().numpy()
        return 1/(1+np.exp(-f))
    def accept_mask(self,X): return self._score(X)<=self.tau
    def max_dev(self,X):
        self.head.eval()
        with torch.no_grad(): r=self.head.residual(torch.tensor(X).to(DEVICE)).cpu().numpy()
        return float(np.abs(r).max())
print("envelope ready")

## 6. Stage A: ADCD per-box labels, padded crops, embeddings, dataset flow
Check the printed ADCD class names for ids 4, 6, 7. If any line says CHECK THIS, stop.

In [ ]:
# =====================================================================================
# STAGE A - ADCD per-box labels, padded crops, image-grouped splits, dataset flow
# -------------------------------------------------------------------------------------
# Needs sections 1-5 of this notebook.
#
# Labeling and crop protocol:
#   1. OLD: every ADCD crop got the IMAGE-level label (all conditions anywhere in the
#      radiograph). 69-74% of crops carried 2-3 labels.
#      NEW: each box gets its OWN class. Boxes of other classes that overlap it with
#      IoU >= ADCD_MERGE_IOU are treated as the same region and added (multi-label).
#   2. OLD: ADCD crops had no margin, DENTEX crops had a 15% margin.
#      NEW: ADCD crops use crop_tooth(..., pad=PAD), the same margin as DENTEX.
#      This needs a fresh ADCD embedding (new cache files emb_adcd_v2_<backbone>.npy).
#   3. OLD: no held-out ADCD split for the ladder; image groups were reconstructed later.
#      NEW: image index is recorded per crop at build time; adcd_split_v2(seed) gives a
#      70/15/15 image-grouped split used by every later V2 cell.
#
# Also writes:
#   v2/dataset_flow_v2.csv      images, crops, exclusions, positives by class and split
#   v2/adcd_class_check.json    ADCD class names for ids 4/6/7 read from the dataset files
#   v2/adcd_site_probe.json     folder / filename-prefix structure (to look for centres)
#   v2/crop_size_stats.json     crop and image resolution summaries
#   v2/env.json                 hardware / software log
#   v2/Figure_crop_examples.png/.pdf   representative crops per condition and dataset
# =====================================================================================
import os, json, glob, platform, re, csv, time
import numpy as np
from collections import Counter, defaultdict
from PIL import Image

V2 = os.path.join(OUT2, V2_NAME); os.makedirs(V2, exist_ok=True)
V2_BACKBONES   = ["dental_vitl", "biomedclip", "dinov3"]   # drop "dinov3" if not available
ADCD_PAD       = PAD          # same 15% margin as DENTEX crop_tooth
ADCD_MERGE_IOU = 0.5          # boxes of different classes overlapping this much = same region
# Crop construction: ADCD boxes are LESION boxes (caries ~80x90 px) while DENTEX boxes are
# TOOTH boxes (~170x290 px). "tooth_context" expands every ADCD box, about its centre, to at least the median
# DENTEX tooth-box size before the 15% margin, so both datasets present tooth-level context. "lesion_box"
# reproduces the earlier behaviour. The annotation box itself is kept for the geometry baseline.
ADCD_CROP_MODE = globals().get("ADCD_CROP_MODE", "tooth_context")  # configured in section 2, part of output path
ADCD_CACHE_TAG = "v2tc" if ADCD_CROP_MODE == "tooth_context" else "v2"
ADCD_MIN_SIDE  = 8            # same skip rule as the old parser
REEMBED_V2   = False        # embeddings are label-independent: computed once, reused by later runs
EXPECTED = {4: ("car",), 6: ("imp", "imt"), 7: ("api", "peri")}   # sanity substrings

# ---------- environment log ----------
def _env_log():
    import torch
    info = {"platform": platform.platform(), "machine": platform.machine(),
            "processor": platform.processor(), "python": platform.python_version(),
            "torch": torch.__version__, "device": str(DEVICE)}
    try:
        info["ram_gb"] = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1)
    except Exception:
        pass
    try:
        import subprocess
        out = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True)
        if out.stdout.strip(): info["cpu"] = out.stdout.strip()
    except Exception:
        pass
    json.dump(info, open(os.path.join(V2, "env.json"), "w"), indent=2)
    print("env:", info)
_env_log()

# ---------- ADCD class-name check ----------
def adcd_class_check():
    names = None; src = None
    for pat in ("**/data.yaml", "**/dataset.yaml", "**/*.yaml", "**/classes.txt"):
        for p in glob.glob(os.path.join(ADCD_DIR, pat), recursive=True):
            txt = open(p, errors="ignore").read()
            if p.endswith(".txt"):
                cand = [l.strip() for l in txt.splitlines() if l.strip()]
            else:
                m = re.search(r"names\s*:\s*\[(.*?)\]", txt, flags=re.S)
                if m:
                    cand = [s.strip().strip("'\"") for s in m.group(1).split(",") if s.strip()]
                else:
                    cand = re.findall(r"^\s*(\d+)\s*:\s*['\"]?([^'\"\n]+)", txt, flags=re.M)
                    cand = [v for _, v in sorted(cand, key=lambda t: int(t[0]))] if cand else None
            if cand and len(cand) > 7:
                names, src = cand, p; break
        if names: break
    out = {"source_file": src, "names": names, "used_ids": {}}
    if names:
        for cid, dx in ADCD_CLASS_TO_DX.items():
            nm = names[cid] if cid < len(names) else None
            ok = nm is not None and any(s in nm.lower() for s in EXPECTED.get(cid, ()))
            out["used_ids"][str(cid)] = {"mapped_to": dx, "dataset_name": nm, "looks_right": bool(ok)}
            print(f"ADCD id {cid} -> {dx:18s} dataset name: {nm!r}  {'OK' if ok else 'CHECK THIS'}")
    else:
        print("ADCD class-name file not found. Verify ids 4/6/7 by hand before trusting results.")
    json.dump(out, open(os.path.join(V2, "adcd_class_check.json"), "w"), indent=2)
    return out
ADCD_CLASS_CHECK = adcd_class_check()
if ADCD_CLASS_CHECK.get("names") and any(not v.get("looks_right") for v in ADCD_CLASS_CHECK["used_ids"].values()):
    raise RuntimeError("ADCD class IDs do not match the labels in its dataset definition; correct the class map before running models")

# ---------- per-box parser ----------
def _xyxy_iou(a, b):
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1]); ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix1 - ix0), max(0.0, iy1 - iy0); inter = iw * ih
    u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / u if u > 0 else 0.0

def parse_adcd_v2():
    """One record per valid relevant box: image path, image index, class, xyxy box, label vector."""
    # ADCD layout: <root>/{train,valid,test}/{images,labels}/ and <root>/test_alte_cabinete/Ext-validation/{images,labels}/
    # Each label file is matched to the image with the same stem in the SIBLING images folder, because file
    # names can repeat across subsets. The external set (other dental offices) is flagged and held out.
    label_files = []
    for d, _, fs in os.walk(ADCD_DIR):
        if "labels" in [q.lower() for q in os.path.relpath(d, ADCD_DIR).split(os.sep)]:
            label_files += [os.path.join(d, f) for f in fs if f.lower().endswith(".txt")]
    label_files = sorted(label_files)
    stems_seen = Counter(os.path.splitext(os.path.basename(f))[0] for f in label_files)
    dup = sum(1 for v in stems_seen.values() if v > 1)
    if dup: print(f"note: {dup} label file names repeat across subsets; each is matched within its own folder")
    stats = Counter(); recs = []; images = []
    for lf in label_files:
        stats["label_files"] += 1
        stem = os.path.splitext(os.path.basename(lf))[0]
        parts_ = os.path.dirname(lf).split(os.sep)
        li = max(i for i, q in enumerate(parts_) if q.lower() == "labels")
        img_dir = os.sep.join(parts_[:li] + [parts_[li].replace("labels", "images").replace("Labels", "Images")] + parts_[li+1:])
        ip = None
        for ext in (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"):
            if os.path.exists(os.path.join(img_dir, stem + ext)): ip = os.path.join(img_dir, stem + ext); break
        if ip is None: stats["label_file_without_image"] += 1; continue
        rel = os.path.relpath(lf, ADCD_DIR).lower()
        subset = rel.split(os.sep)[0]
        external = ("alte" in rel) or ("ext-validation" in rel) or ("ext_validation" in rel)
        W, H = Image.open(ip).size
        boxes = []
        for line in open(lf):
            parts = line.split()
            if len(parts) < 5: continue
            cid = int(float(parts[0])); stats[f"box_class_{cid}"] += 1
            cond = ADCD_CLASS_TO_DX.get(cid)
            if cond is None: continue
            cx, cy, w, h = map(float, parts[1:5])
            x0, y0, x1, y1 = (cx-w/2)*W, (cy-h/2)*H, (cx+w/2)*W, (cy+h/2)*H
            x0, x1 = min(x0, x1), max(x0, x1); y0, y1 = min(y0, y1), max(y0, y1)
            stats["relevant_boxes"] += 1
            if (x1-x0) < ADCD_MIN_SIDE or (y1-y0) < ADCD_MIN_SIDE or x1 <= 0 or y1 <= 0 or x0 >= W or y0 >= H:
                stats["relevant_boxes_skipped_small_or_outside"] += 1; continue
            boxes.append((cond, [x0, y0, x1, y1]))
        if not boxes:
            stats["images_without_relevant_box"] += 1; continue
        img_idx = len(images); images.append({"path": ip, "W": W, "H": H, "subset": subset, "external": external})
        stats["images_used_" + ("external" if external else "internal")] += 1
        for i, (cond, b) in enumerate(boxes):
            y = np.zeros(NUM_DX, np.float32); y[CLASS_IDX[cond]] = 1.0
            merged = []
            for j, (cond2, b2) in enumerate(boxes):
                if j != i and cond2 != cond and _xyxy_iou(b, b2) >= ADCD_MERGE_IOU:
                    y[CLASS_IDX[cond2]] = 1.0; merged.append(cond2)
            if merged: stats["boxes_with_merged_label"] += 1
            recs.append({"img_idx": img_idx, "path": ip, "cls": cond, "xyxy": b, "subset": subset, "external": bool(external), "W": W, "H": H,
                         "xywh": [b[0], b[1], b[2]-b[0], b[3]-b[1]], "y": y})
    stats["images_used"] = len(images); stats["crops"] = len(recs)
    return recs, images, dict(stats)

ADCD_RECS, ADCD_IMAGES, ADCD_PARSE_STATS = parse_adcd_v2()
ADCD_Y      = np.stack([r["y"] for r in ADCD_RECS]).astype(np.float32)
import hashlib
def _md5f(p):
    h = hashlib.md5()
    with open(p, "rb") as fh:
        for ch in iter(lambda: fh.read(1 << 20), b""): h.update(ch)
    return h.hexdigest()
_img_hash = [_md5f(im["path"]) for im in ADCD_IMAGES]
_first = {}
_hash_group = [_first.setdefault(hh, i) for i, hh in enumerate(_img_hash)]   # identical files -> same group id
_n_dup = len(_img_hash) - len(set(_img_hash))
ADCD_PARSE_STATS["duplicate_radiograph_files"] = int(_n_dup)
print(f"ADCD radiographs with byte-identical duplicates elsewhere in the release: {_n_dup}"
      + ("  (grouped together so they never fall in different splits)" if _n_dup else ""))
ADCD_GROUPS = np.array([_hash_group[r["img_idx"]] for r in ADCD_RECS], dtype=int)
ADCD_EXT    = np.array([r["external"] for r in ADCD_RECS], dtype=bool)
ADCD_EXT_IDX = np.where(ADCD_EXT)[0]
# Internal copies of EXTERNAL radiographs are removed from the internal pool, so the external set stays unseen.
_ext_hashes = {_img_hash[i] for i, im in enumerate(ADCD_IMAGES) if im["external"]}
_int_dup_imgs = [i for i, im in enumerate(ADCD_IMAGES) if (not im["external"]) and _img_hash[i] in _ext_hashes]
ADCD_EXCL = np.array([(not r["external"]) and (_img_hash[r["img_idx"]] in _ext_hashes) for r in ADCD_RECS], dtype=bool)
_dup_dirs = Counter(os.path.relpath(os.path.dirname(ADCD_IMAGES[i]["path"]), ADCD_DIR) for i in _int_dup_imgs)
_int_hashes = [_img_hash[i] for i, im in enumerate(ADCD_IMAGES) if not im["external"]]
ADCD_PARSE_STATS.update({"external_radiographs": int(sum(im["external"] for im in ADCD_IMAGES)),
                         "external_radiographs_also_in_internal_folders": int(sum(1 for h in _ext_hashes if h in set(_int_hashes))),
                         "internal_copies_of_external_removed": len(_int_dup_imgs),
                         "internal_internal_duplicate_files": int(len(_int_hashes) - len(set(_int_hashes))),
                         "internal_crops_removed_as_external_copies": int(ADCD_EXCL.sum())})
print(f"external radiographs: {ADCD_PARSE_STATS['external_radiographs']} | of these also present in train/valid/test folders: "
      f"{ADCD_PARSE_STATS['external_radiographs_also_in_internal_folders']} | internal copies removed from the internal pool: "
      f"{len(_int_dup_imgs)} ({int(ADCD_EXCL.sum())} crops) | folders: {dict(_dup_dirs)} | internal-internal duplicates: "
      f"{ADCD_PARSE_STATS['internal_internal_duplicate_files']}")
# tooth-context crop boxes
_MED_W = float(np.median([t["bbox"][2] for t in teeth])); _MED_H = float(np.median([t["bbox"][3] for t in teeth]))
for r in ADCD_RECS:
    x0, y0, w, h = r["xywh"]
    if ADCD_CROP_MODE == "tooth_context":
        w2, h2 = max(w, _MED_W), max(h, _MED_H); cx, cy = x0 + w / 2, y0 + h / 2
        nx0 = min(max(0.0, cx - w2 / 2), max(0.0, r["W"] - w2)); ny0 = min(max(0.0, cy - h2 / 2), max(0.0, r["H"] - h2))
        r["xywh_crop"] = [nx0, ny0, min(w2, r["W"]), min(h2, r["H"])]
    else:
        r["xywh_crop"] = [x0, y0, w, h]
ADCD_PARSE_STATS.update({"crop_mode": ADCD_CROP_MODE, "min_crop_w_px": _MED_W, "min_crop_h_px": _MED_H})
print(f"ADCD crop mode: {ADCD_CROP_MODE} (minimum box {_MED_W:.0f} x {_MED_H:.0f} px before the 15% margin)")
print(f"ADCD crops parsed: {int((~ADCD_EXT).sum())} in the train/valid/test folders (before removing copies of external "
      f"radiographs, see next line) | {int(ADCD_EXT.sum())} external (other centres, held out as a second target)")
print("ADCD parse:", ADCD_PARSE_STATS)
k_per = ADCD_Y[:, [CLASS_IDX[c] for c in TARGET_CLASSES]].sum(1).astype(int)
print("labels per crop:", dict(Counter(k_per.tolist())), "| prevalence:",
      {c: round(float(ADCD_Y[:, CLASS_IDX[c]].mean()), 3) for c in TARGET_CLASSES})

# persist (so later cells survive a kernel restart)
np.save(os.path.join(V2, "adcd_labels_v2.npy"), ADCD_Y)
np.save(os.path.join(V2, "adcd_groups_v2.npy"), ADCD_GROUPS)
np.save(os.path.join(V2, "adcd_external_v2.npy"), ADCD_EXT)
np.save(os.path.join(V2, "adcd_excluded_v2.npy"), ADCD_EXCL)
json.dump({"crop_mode": ADCD_CROP_MODE, "cache_tag": ADCD_CACHE_TAG}, open(os.path.join(V2, "adcd_crop_mode.json"), "w"))
json.dump(ADCD_PARSE_STATS, open(os.path.join(V2, "adcd_parse_stats.json"), "w"), indent=2)
json.dump([{k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in r.items()} for r in ADCD_RECS],
          open(os.path.join(V2, "adcd_records_v2.json"), "w"))
json.dump(ADCD_PARSE_STATS, open(os.path.join(V2, "adcd_parse_stats.json"), "w"), indent=2)

# ---------- splits ----------
def grouped_split_by_array(groups, seed, vf=0.15, tf=0.15):
    gids = np.unique(groups); rng = np.random.RandomState(seed); rng.shuffle(gids)
    n = len(gids); nte = int(n*tf); nva = int(n*vf)
    te_g, va_g = set(gids[:nte].tolist()), set(gids[nte:nte+nva].tolist())
    tr, va, te = [], [], []
    for i, g in enumerate(groups):
        (te if g in te_g else va if g in va_g else tr).append(i)
    return np.array(tr), np.array(va), np.array(te)

def adcd_split_v2(seed):
    """70/15/15 image-grouped split of the INTERNAL ADCD crops; external crops and internal copies of
    external radiographs never enter it."""
    main = np.where(~ADCD_EXT & ~ADCD_EXCL)[0]
    tr, va, te = grouped_split_by_array(ADCD_GROUPS[main], seed)
    return main[tr], main[va], main[te]

def dentex_groups():
    return np.array([t["image_id"] for t in teeth])

# ---------- site probe (is there any centre information in the release?) ----------
def adcd_site_probe():
    rel_dirs = Counter(os.path.relpath(os.path.dirname(im["path"]), ADCD_DIR) for im in ADCD_IMAGES)
    stems = [os.path.splitext(os.path.basename(im["path"]))[0] for im in ADCD_IMAGES]
    prefixes = Counter(re.split(r"[_\-\s\.]", s)[0][:6] for s in stems)
    sizes = Counter(f"{im['W']}x{im['H']}" for im in ADCD_IMAGES)
    out = {"image_dirs": dict(rel_dirs), "filename_prefix_top20": dict(prefixes.most_common(20)),
           "image_size_top20": dict(sizes.most_common(20)), "n_distinct_sizes": len(sizes)}
    json.dump(out, open(os.path.join(V2, "adcd_site_probe.json"), "w"), indent=2)
    print("ADCD image dirs:", dict(rel_dirs))
    print("ADCD image sizes (top 5):", sizes.most_common(5), "| distinct sizes:", len(sizes))
    print("If image size or filename prefix tracks the acquiring centre, a centre-wise split could be built from it.")
    return out
ADCD_SITE_PROBE = adcd_site_probe()

# ---------- padded crops + embeddings ----------
def adcd_crop_v2(i, _cache={}):
    r = ADCD_RECS[i]; p = r["path"]
    if p not in _cache:
        if len(_cache) > 64: _cache.clear()
        _cache[p] = Image.open(p).convert("RGB")
    return crop_tooth(_cache[p], r["xywh_crop"], pad=ADCD_PAD)

make_embedder_v2 = make_embedder      # Dental ViT-L input size is read from the checkpoint in section 4

def _embed_list(name, n, crop_fn, fp, tag):
    if os.path.exists(fp) and not REEMBED_V2:
        X = np.load(fp); assert len(X) == n, f"{fp}: {len(X)} rows vs {n} crops"; return X
    emb = make_embedder_v2(name); t0 = time.time(); X = []; B = 256
    for k in range(0, n, B):
        X.append(emb([crop_fn(i) for i in range(k, min(k+B, n))]))
        print(f"  {tag}/{name}: {min(k+B, n)}/{n} crops ({time.time()-t0:.0f}s)", end="\r")
    X = np.concatenate(X, 0).astype(np.float32); np.save(fp, X)
    print(f"\n  saved {os.path.basename(fp)} {X.shape} in {(time.time()-t0)/60:.1f} min")
    return X

def embed_v2(name):
    """Re-embed BOTH datasets with the identical embedder and margin, so source and target
    differ only in the images, not in preprocessing."""
    _embed_list(name, len(teeth), get_crop, os.path.join(CACHE, f"emb_dentex_v2_{name}.npy"), "DENTEX v2")
    _embed_list(name, len(ADCD_RECS), adcd_crop_v2, os.path.join(CACHE, f"emb_adcd_{ADCD_CACHE_TAG}_{name}.npy"), "ADCD " + ADCD_CACHE_TAG)

def load_adcd_v2(name):
    fp = os.path.join(CACHE, f"emb_adcd_{ADCD_CACHE_TAG}_{name}.npy")
    X = np.load(fp); assert len(X) == len(ADCD_Y), f"{fp}: {len(X)} rows vs {len(ADCD_Y)} labels"
    return X

def load_dentex_v2(name):
    fp = os.path.join(CACHE, f"emb_dentex_v2_{name}.npy")
    X = np.load(fp); assert len(X) == len(teeth), f"{fp}: {len(X)} rows vs {len(teeth)} teeth"
    return X

EMB_TIMES = {}
for b in list(V2_BACKBONES):
    t0 = time.time()
    try:
        embed_v2(b); EMB_TIMES[b] = round((time.time()-t0)/60, 2)
    except Exception as e:
        print(f"  {b}: embedding failed ({type(e).__name__}: {e}); removed from V2_BACKBONES")
        V2_BACKBONES.remove(b)
json.dump(EMB_TIMES, open(os.path.join(V2, "embedding_minutes.json"), "w"), indent=2)

# ---------- dataset flow table ----------
def dataset_flow():
    rows = []
    dg = dentex_groups()
    def add(ds, split, idx, Ym, groups):
        row = {"dataset": ds, "split": split, "images": int(len(np.unique(groups[idx]))), "crops": int(len(idx))}
        for c in CLASSES:
            if ds == "ADCD" and c == "Deep Caries": continue
            row[f"pos_{c}"] = int(Ym[idx, CLASS_IDX[c]].sum())
        row["crops_with_2plus_labels"] = int((Ym[idx][:, [CLASS_IDX[c] for c in TARGET_CLASSES]].sum(1) >= 2).sum())
        rows.append(row)
    add("DENTEX", "all", np.arange(len(teeth)), Y, dg)
    add("ADCD", "internal_all", np.where(~ADCD_EXT & ~ADCD_EXCL)[0], ADCD_Y, ADCD_GROUPS)
    if ADCD_EXCL.any(): add("ADCD", "internal_removed_copies_of_external", np.where(ADCD_EXCL)[0], ADCD_Y, ADCD_GROUPS)
    if len(ADCD_EXT_IDX): add("ADCD", "external_heldout", ADCD_EXT_IDX, ADCD_Y, ADCD_GROUPS)
    for s in SEEDS:
        tr, va, te = grouped_split(teeth, s)
        for nm, idx in (("train", tr), ("val", va), ("test", te)): add("DENTEX", f"{nm}_seed{s}", np.array(idx), Y, dg)
        tr, va, te = adcd_split_v2(s)
        for nm, idx in (("train", tr), ("val", va), ("test", te)): add("ADCD", f"{nm}_seed{s}", idx, ADCD_Y, ADCD_GROUPS)
    keys = sorted({k for r in rows for k in r}, key=lambda k: (k not in ("dataset", "split", "images", "crops"), k))
    with open(os.path.join(V2, "dataset_flow_v2.csv"), "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=keys); w.writeheader(); w.writerows(rows)
    print("saved dataset_flow_v2.csv")
    for r in rows[:3]: print("  ", r)
dataset_flow()

# ---------- crop / image size stats ----------
def size_stats():
    def summ(a): a = np.asarray(a, float); return {"median": float(np.median(a)), "p10": float(np.percentile(a, 10)), "p90": float(np.percentile(a, 90))}
    dw = [t["bbox"][2]*(1+2*PAD) for t in teeth]; dh = [t["bbox"][3]*(1+2*PAD) for t in teeth]
    aw = [r["xywh"][2]*(1+2*ADCD_PAD) for r in ADCD_RECS]; ah = [r["xywh"][3]*(1+2*ADCD_PAD) for r in ADCD_RECS]   # annotation boxes
    cw = [r["xywh_crop"][2]*(1+2*ADCD_PAD) for r in ADCD_RECS]; ch = [r["xywh_crop"][3]*(1+2*ADCD_PAD) for r in ADCD_RECS]
    out = {"DENTEX_crop_w": summ(dw), "DENTEX_crop_h": summ(dh), "ADCD_crop_w": summ(aw), "ADCD_crop_h": summ(ah),
           "ADCD_actual_crop_w": summ(cw), "ADCD_actual_crop_h": summ(ch)}
    for c in TARGET_CLASSES:
        idx = [i for i, r in enumerate(ADCD_RECS) if r["cls"] == c]
        out[f"ADCD_crop_w_{c}"] = summ([aw[i] for i in idx]); out[f"ADCD_crop_h_{c}"] = summ([ah[i] for i in idx])
        out[f"ADCD_actual_crop_w_{c}"] = summ([cw[i] for i in idx]); out[f"ADCD_actual_crop_h_{c}"] = summ([ch[i] for i in idx])
        didx = [i for i in range(len(teeth)) if Y[i, CLASS_IDX[c]] > 0]
        out[f"DENTEX_crop_w_{c}"] = summ([dw[i] for i in didx]); out[f"DENTEX_crop_h_{c}"] = summ([dh[i] for i in didx])
    json.dump(out, open(os.path.join(V2, "crop_size_stats.json"), "w"), indent=2)
    print("crop sizes (median w x h px): DENTEX", round(out["DENTEX_crop_w"]["median"]), "x", round(out["DENTEX_crop_h"]["median"]),
          "| ADCD", round(out["ADCD_crop_w"]["median"]), "x", round(out["ADCD_crop_h"]["median"]))
    print("  per-class ADCD medians (crop geometry may encode class; compare with DENTEX):")
    for c in TARGET_CLASSES:
        print(f"    {c:18s} ADCD box {out[f'ADCD_crop_w_{c}']['median']:.0f}x{out[f'ADCD_crop_h_{c}']['median']:.0f}"
              f"  ADCD crop used {out[f'ADCD_actual_crop_w_{c}']['median']:.0f}x{out[f'ADCD_actual_crop_h_{c}']['median']:.0f}"
              f"   DENTEX {out[f'DENTEX_crop_w_{c}']['median']:.0f}x{out[f'DENTEX_crop_h_{c}']['median']:.0f}")
size_stats()

# ---------- representative crops ----------
def crop_examples(n=4, seed=0):
    import matplotlib.pyplot as plt
    rng = np.random.RandomState(seed)
    fig, axes = plt.subplots(2*len(TARGET_CLASSES), n, figsize=(2.0*n, 2.0*2*len(TARGET_CLASSES)))
    row = 0
    for c in TARGET_CLASSES:
        d_idx = np.where(Y[:, CLASS_IDX[c]] > 0)[0]; a_idx = np.array([i for i, r in enumerate(ADCD_RECS) if r["cls"] == c])
        for ds, pool in (("DENTEX", d_idx), ("ADCD", a_idx)):
            pick = rng.choice(pool, size=min(n, len(pool)), replace=False) if len(pool) else []
            for j in range(n):
                ax = axes[row, j]; ax.axis("off")
                if j < len(pick):
                    im = get_crop(int(pick[j])) if ds == "DENTEX" else adcd_crop_v2(int(pick[j]))
                    ax.imshow(im.convert("L"), cmap="gray")
                if j == 0: ax.set_title(f"{ds}: {c}", fontsize=8, loc="left")
            row += 1
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(V2, f"Figure_crop_examples.{ext}"), dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig); print("saved Figure_crop_examples.png/.pdf")
crop_examples()
def _log_versions():
    """Package versions, ViT-pytorch commit and Hugging Face snapshot revisions actually used (reproducibility)."""
    info = json.load(open(os.path.join(V2, "env.json")))
    import importlib, subprocess
    pk = {}
    for m in ("numpy", "scipy", "sklearn", "torch", "torchvision", "transformers", "open_clip", "timm", "PIL", "huggingface_hub"):
        try: pk[m] = importlib.import_module(m).__version__
        except Exception: pk[m] = None
    info["packages"] = pk
    try: info["vit_pytorch_commit"] = subprocess.run(["git", "-C", "ViT-pytorch", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    except Exception: pass
    try:
        from huggingface_hub import scan_cache_dir
        info["hf_revisions"] = {r.repo_id: sorted(rv.commit_hash for rv in r.revisions) for r in scan_cache_dir().repos
                                if any(k in r.repo_id for k in ("Teeth_Visual_Experts", "dinov3", "BiomedCLIP"))}
    except Exception as e:
        info["hf_revisions"] = f"unavailable: {e}"
    json.dump(info, open(os.path.join(V2, "env.json"), "w"), indent=2)
    print("versions logged:", info.get("packages"), "| ViT-pytorch", info.get("vit_pytorch_commit"))
_log_versions()
json.dump(V2_BACKBONES, open(os.path.join(V2, "backbones_v2.json"), "w"))
# Stop before the long training stage if any backbone required by the manuscript is unavailable.
_required = {"dental_vitl", "biomedclip", "dinov3"}
_missing = sorted(_required - set(V2_BACKBONES))
if _missing:
    raise RuntimeError("Missing required foundation backbones: " + ", ".join(_missing) +
                       ". Fix model access/loading now rather than silently running an incomplete experiment.")
print("\nCell A done. Later V2 cells use: ADCD_RECS, ADCD_Y, ADCD_GROUPS, adcd_split_v2(),")
print("load_adcd_v2(), load_dentex_v2(), V2_BACKBONES.")

## 7. Common evaluation helpers

In [ ]:
# =====================================================================================
# V2 COMMON - shared helpers for cells B, C, D, E
# -------------------------------------------------------------------------------------
# After a kernel restart: run sections 1-5 and then this cell; it reloads Stage A's
# outputs from disk, so Stage A does not need to be repeated.
#
# Evaluation conventions used by every V2 result (fixes the audit findings):
#   * Target metrics are computed on the ADCD TEST split only (image-grouped, 15%).
#     Labeled target slices are drawn from the ADCD TRAIN split only.
#   * Source metrics are computed on the DENTEX TEST split (the old code used the
#     validation split, which also sets the temperature and the gate threshold).
#   * Every target result is reported three ways:
#       full  : all test crops (common population, no gate)
#       src   : crops accepted by the source-anchored gate (mu, Sigma, tau from DENTEX)
#       tgt   : crops accepted by a target-fitted gate (mu, Sigma from unlabeled ADCD
#               train embeddings, tau = 95th percentile on unlabeled ADCD val)
#     plus matched coverage (the 20% and 50% most in-distribution test crops under the
#     source gate score), coverage, and accepted positives per class.
#   * The ADCD external set (test_alte_cabinete/Ext-validation, other dental offices) is never used
#     for training, slices or gates; every model is also scored on it ("tgt_ext"), as a centre-wise check.
#   * ECE uses 15 equal-width bins (the manuscript states 15; the old code used 10).
#   * Per-tooth scores with image ids are saved for every run, for cluster bootstrap.
# =====================================================================================
import os, json, copy, csv, time
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

V2 = os.path.join(OUT2, V2_NAME); os.makedirs(V2, exist_ok=True)
PT_DIR = os.path.join(V2, "per_tooth"); os.makedirs(PT_DIR, exist_ok=True)
TCOLS = [CLASS_IDX[c] for c in TARGET_CLASSES]
MATCHED_COVERAGE = (0.2, 0.5)
ECE_BINS = 15

# ---------- reload Cell A outputs if needed ----------
if "ADCD_Y" not in globals():
    ADCD_Y = np.load(os.path.join(V2, "adcd_labels_v2.npy"))
    ADCD_GROUPS = np.load(os.path.join(V2, "adcd_groups_v2.npy"))
    ADCD_EXT = np.load(os.path.join(V2, "adcd_external_v2.npy"))
    _ex = os.path.join(V2, "adcd_excluded_v2.npy")
    ADCD_EXCL = np.load(_ex) if os.path.exists(_ex) else np.zeros(len(ADCD_EXT), bool)
    ADCD_CACHE_TAG = json.load(open(os.path.join(V2, "adcd_crop_mode.json")))["cache_tag"] if os.path.exists(os.path.join(V2, "adcd_crop_mode.json")) else "v2"
    ADCD_RECS = json.load(open(os.path.join(V2, "adcd_records_v2.json")))
    for r in ADCD_RECS: r["y"] = np.asarray(r["y"], np.float32)
    V2_BACKBONES = json.load(open(os.path.join(V2, "backbones_v2.json")))
    ADCD_PAD = PAD
    print("reloaded Cell A outputs:", ADCD_Y.shape, "backbones:", V2_BACKBONES)

def grouped_split_by_array(groups, seed, vf=0.15, tf=0.15):
    gids = np.unique(groups); rng = np.random.RandomState(seed); rng.shuffle(gids)
    n = len(gids); nte = int(n*tf); nva = int(n*vf)
    te_g, va_g = set(gids[:nte].tolist()), set(gids[nte:nte+nva].tolist())
    tr, va, te = [], [], []
    for i, g in enumerate(groups):
        (te if g in te_g else va if g in va_g else tr).append(i)
    return np.array(tr), np.array(va), np.array(te)

def adcd_split_v2(seed):
    """70/15/15 image-grouped split of the INTERNAL ADCD crops; external crops never enter it."""
    main = np.where(~ADCD_EXT & ~ADCD_EXCL)[0]
    tr, va, te = grouped_split_by_array(ADCD_GROUPS[main], seed)
    return main[tr], main[va], main[te]

ADCD_EXT_IDX = np.where(ADCD_EXT)[0]
HAS_EXT = len(ADCD_EXT_IDX) >= 20
if not HAS_EXT:
    raise RuntimeError(f"Only {len(ADCD_EXT_IDX)} external ADCD crops were found. "
                       "Check ADCD_DIR/test_alte_cabinete/Ext-validation/{images,labels} before the expensive run.")

def dentex_split_v2(seed):
    tr, va, te = grouped_split(teeth, seed)
    return np.array(tr), np.array(va), np.array(te)

DENTEX_GROUPS = np.array([t["image_id"] for t in teeth])

def load_adcd_v2(name):
    X = np.load(os.path.join(CACHE, f"emb_adcd_{ADCD_CACHE_TAG}_{name}.npy")); assert len(X) == len(ADCD_Y); return X

def load_dentex_v2(name):
    X = np.load(os.path.join(CACHE, f"emb_dentex_v2_{name}.npy")); assert len(X) == len(teeth); return X

def label_slice(atr, budget, seed):
    """Nested slices: the 16-crop slice is contained in the 32-crop slice, and so on."""
    perm = np.random.RandomState(seed).permutation(len(atr))
    return atr[perm[:budget]]

# ---------- metrics ----------
def auc_safe(y, p):
    y = np.asarray(y); return float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else float("nan")

def ece15(y, p, nb=ECE_BINS):
    b = np.linspace(0, 1, nb+1); e = 0.0
    for lo, hi in zip(b[:-1], b[1:]):
        m = (p > lo) & (p <= hi)
        if m.sum(): e += (m.sum()/len(p)) * abs(p[m].mean() - y[m].mean())
    return float(e)

def _block(P, Yt, mask, names=TARGET_CLASSES):
    """Per-condition AUROC, AUPRC, prevalence and counts. The macro is defined only when EVERY condition
    is evaluable (both classes present, >= 10 crops); otherwise it is NaN and macro_n_valid says why."""
    out = {"n": int(mask.sum()), "coverage": float(mask.mean()) if len(mask) else float("nan")}
    aucs, eces = [], []
    for j, c in enumerate(names):
        yy, pp = Yt[mask, j], P[mask, j]
        ok = mask.sum() >= 10 and yy.sum() >= 5 and (len(yy) - yy.sum()) >= 5     # >= 5 positives and >= 5 negatives
        a = auc_safe(yy, pp) if ok else float("nan")
        out[f"auc_{c}"] = a; out[f"pos_{c}"] = int(yy.sum()); out[f"neg_{c}"] = int(len(yy) - yy.sum())
        out[f"auprc_{c}"] = float(average_precision_score(yy, pp)) if ok else float("nan")
        out[f"prev_{c}"] = float(yy.mean()) if len(yy) else float("nan")
        if ok: aucs.append(a); eces.append(ece15(yy, pp))
    out["macro_n_valid"] = len(aucs)
    out["macro"] = float(np.mean(aucs)) if len(aucs) == len(names) else float("nan")
    out["ece"] = float(np.mean(eces)) if len(eces) == len(names) else float("nan")
    return out

# ---------- gates ----------
def fit_target_gate(X_unlab_tr, X_unlab_va):
    score, mu, ci = maha_scorer(X_unlab_tr)
    tau = float(np.quantile(score(X_unlab_va), OOD_Q))
    return {"score": score, "tau": tau}

def with_gate(env, gate_from):
    """Copy of env whose Mahalanobis gate is taken from another envelope."""
    e = copy.copy(env)
    e._score, e.mu, e.ci, e.tau = gate_from._score, gate_from.mu, gate_from.ci, gate_from.tau
    return e

def with_tau(env, tau):
    e = copy.copy(env); e.tau = float(tau); return e

# ---------- full evaluation of one adapted model on the ADCD test split ----------
def eval_target_v2(env, Xt, Yfull, groups, tgt_gate=None, fixed_score=None):
    """fixed_score: Mahalanobis score of the SAME test crops under the unadapted source gate. Its 20%/50%
    most typical crops form one fixed tooth set shared by every technique (views fix20/fix50), so
    paired comparisons use identical teeth. cov20/cov50 instead use each model's own gate ordering."""
    P = env.proba(Xt)[:, TCOLS]; Yt = Yfull[:, TCOLS]
    m_src = env._score(Xt); acc_src = m_src <= env.tau
    res = {"full": _block(P, Yt, np.ones(len(Yt), bool)), "src": _block(P, Yt, acc_src),
           "max_dev_tgt": env.max_dev(Xt)}
    order = np.argsort(m_src)
    for f in MATCHED_COVERAGE:
        mk = np.zeros(len(Yt), bool); mk[order[:max(1, int(round(f*len(Yt))))]] = True
        res[f"cov{int(f*100)}"] = _block(P, Yt, mk)
    if fixed_score is not None:
        fo = np.argsort(fixed_score)
        for f in MATCHED_COVERAGE:
            mk = np.zeros(len(Yt), bool); mk[fo[:max(1, int(round(f*len(Yt))))]] = True
            res[f"fix{int(f*100)}"] = _block(P, Yt, mk)
    m_tgt = None
    if tgt_gate is not None:
        m_tgt = tgt_gate["score"](Xt); res["tgt"] = _block(P, Yt, m_tgt <= tgt_gate["tau"])
    arrays = {"probs": P.astype(np.float32), "labels": Yt.astype(np.int8), "groups": np.asarray(groups),
              "maha_src": m_src.astype(np.float32), "tau_src": np.float32(env.tau)}
    if m_tgt is not None:
        arrays["maha_tgt"] = m_tgt.astype(np.float32); arrays["tau_tgt"] = np.float32(tgt_gate["tau"])
    if fixed_score is not None:
        arrays["maha_fixed"] = np.asarray(fixed_score, np.float32)
    return res, arrays

def head_block(P3, Yfull):
    """Full-coverage metrics for a probability matrix over the three shared conditions (control heads)."""
    return _block(np.asarray(P3), Yfull[:, TCOLS], np.ones(len(Yfull), bool))

def eval_ext_v2(env, fn_t, Xa, tgt_gate=None):
    """Same four views on the held-out external ADCD set (other dental offices). Returns (res, arrays) or (None, None)."""
    if not HAS_EXT: return None, None
    return eval_target_v2(env, fn_t(Xa[ADCD_EXT_IDX]), ADCD_Y[ADCD_EXT_IDX], ADCD_GROUPS[ADCD_EXT_IDX], tgt_gate)

def eval_source_v2(env, Xs, Ys, tgt_gate=None):
    """DENTEX test metrics. With tgt_gate, also the refusal of DENTEX teeth by the target-fitted gate
    (the fourth cell of the source/target x source-gate/target-gate matrix)."""
    P = env.proba(Xs); acc = env.accept_mask(Xs)
    full4 = _block(P, Ys, np.ones(len(Ys), bool), CLASSES)
    full3 = _block(P[:, TCOLS], Ys[:, TCOLS], np.ones(len(Ys), bool))
    acc4 = _block(P, Ys, acc, CLASSES)
    out = {"macro4_full": full4["macro"], "macro3_full": full3["macro"], "macro4_acc": acc4["macro"],
           "refusal": float(1 - acc.mean()), "max_dev": env.max_dev(Xs)}
    for c in CLASSES: out[f"auc_{c}"] = full4[f"auc_{c}"]
    if tgt_gate is not None:
        out["tgt_gate_refusal"] = float(np.mean(tgt_gate["score"](Xs) > tgt_gate["tau"]))
    return out

def flatten(rec):
    out = {}
    for k, v in rec.items():
        if isinstance(v, dict):
            for k2, v2 in v.items(): out[f"{k}.{k2}"] = v2
        else:
            out[k] = v
    return out

def save_rows(rows, name):
    json.dump(rows, open(os.path.join(V2, f"{name}.json"), "w"), indent=2, default=str)
    flat = [flatten(r) for r in rows]; keys = sorted({k for r in flat for k in r})
    with open(os.path.join(V2, f"{name}.csv"), "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=keys); w.writeheader(); w.writerows(flat)

def save_per_tooth(tag, arrays):
    np.savez_compressed(os.path.join(PT_DIR, f"{tag}.npz"), **arrays)

print("V2 common helpers ready.")

## 7b. Checks: embedding-cache integrity and duplicate radiographs
Re-embeds 16 fixed crops per dataset and backbone and compares them with the cache, then checks for byte-identical radiographs across ADCD folders and splits.

In [ ]:
# =====================================================================================
# CHECKS - embedding-cache integrity and duplicate radiographs
#   1. For every backbone, 16 fixed DENTEX crops and 16 fixed ADCD crops are re-embedded now and compared
#      with the cached rows. Any cosine below 0.999 means the cache does not match the crops: stop.
#   2. ADCD radiographs are hashed (MD5). Byte-identical files in different folders must never fall in
#      different splits, or in both the internal pool and the external set.
# =====================================================================================
import hashlib, json, os, numpy as np
from collections import defaultdict
from PIL import Image as _PI
def _adcd_crop_chk(i):
    r = ADCD_RECS[i]; return crop_tooth(_PI.open(r["path"]).convert("RGB"), r.get("xywh_crop", r["xywh"]), pad=PAD)
CHK = {"cache": {}, "duplicates": {}}
_bad = []
for b in V2_BACKBONES:
    emb = make_embedder(b)
    for name, n, crop_fn, fname in (("DENTEX", len(teeth), get_crop, f"emb_dentex_v2_{b}.npy"),
                                    ("ADCD", len(ADCD_Y), _adcd_crop_chk, f"emb_adcd_{ADCD_CACHE_TAG}_{b}.npy")):
        Xc = np.load(os.path.join(CACHE, fname)); idx = np.linspace(0, n - 1, 16).astype(int)
        Z = emb([crop_fn(int(i)) for i in idx]); C = Xc[idx]
        cos = (Z * C).sum(1) / (np.linalg.norm(Z, axis=1) * np.linalg.norm(C, axis=1) + 1e-12)
        CHK["cache"][f"{name}/{b}"] = float(cos.min())
        print(f"cache check {name:6s} {b:11s}: min cosine {cos.min():.5f}")
        if cos.min() < 0.999: _bad.append(fname)
paths = sorted({r["path"] for r in ADCD_RECS})
def _md5(p):
    h = hashlib.md5()
    with open(p, "rb") as fh:
        for ch in iter(lambda: fh.read(1 << 20), b""): h.update(ch)
    return h.hexdigest()
hsh = {p: _md5(p) for p in paths}
by_hash = defaultdict(set)
for r, g in zip(ADCD_RECS, ADCD_GROUPS): by_hash[hsh[r["path"]]].add((int(g), bool(r["external"])))
dups = {h: v for h, v in by_hash.items() if len({g for g, _ in v}) > 1}
cross_ext = sum(1 for v in dups.values() if len({e for _, e in v}) > 1)
split_leaks = 0
for s_ in SEEDS:
    tr_, va_, te_ = adcd_split_v2(s_); where = {}
    for nm, ix in (("train", tr_), ("val", va_), ("test", te_)):
        for g in set(ADCD_GROUPS[ix].tolist()): where[g] = nm
    for v in dups.values():
        if len({where.get(g, "external") for g, _ in v}) > 1: split_leaks += 1
CHK["duplicates"] = {"radiograph_files": len(paths), "duplicate_groups_not_merged": len(dups),
                     "internal_external_duplicates": cross_ext, "split_leaks_over_seeds": split_leaks}
print("duplicate check:", CHK["duplicates"])
json.dump(CHK, open(os.path.join(V2, "checks.json"), "w"), indent=2)
if _bad:
    raise RuntimeError(f"Embedding cache does not match the crops for {_bad}. Delete these files and run Stage A again.")
if split_leaks or cross_ext:
    print("WARNING: identical radiographs appear in different splits or in both internal and external sets. Resolve this before using the results.")

## 8. Stage B: frozen-backbone rungs and embedding-space corrections

In [ ]:
# =====================================================================================
# STAGE B - frozen-backbone rungs and embedding-space corrections
# -------------------------------------------------------------------------------------
# Needs: Stage A (or section 7 after a restart).
#
# Technique list. Names now say what the code does (audit findings in brackets):
#   source_only      unadapted source envelope                      [old R0/R1 were this]
#   R0_thresh        label-free: gate threshold tau re-estimated as the 95th percentile
#                    of Mahalanobis distance on UNLABELED ADCD train   [old R0 never did]
#   R0_moment        label-free: target embeddings re-standardised to source mean/SD
#                    using UNLABELED ADCD train statistics            [old "ln_affine"]
#   R1_head          probes, temperature and residual head re-fitted on the labeled ADCD
#                    slice; source-anchored gate kept                  [old R1 never did]
#   E_bias           learned additive vector on the embedding         [old "bitfit"]
#   E_lowrank_r{r}   z + zAB, rank r in {2,4,8,16}                     [old "lora"; old
#                    "vpt" was rank 2 and old "adapter" was rank 8, so both are covered]
#   E_linear_l2      full d x d correction, L2 penalty to its initial value
#                    (identity Fisher; NOT EWC)                        [old "full_ewc"]
#   E_linear_replay  full d x d correction trained on the ADCD slice WITH replay of
#                    DENTEX train embeddings                           [old "full_rehearsal"
#                    used no source teeth]
#   E_linear_avg     z + 0.5 * correction (average of frozen and corrected embedding)
#                                                                      [old "dual_encoder"]
# For E_* techniques the envelope is re-fitted on the corrected DENTEX embeddings, exactly
# as in the original design; the correction itself is trained on the ADCD slice.
# All losses use the three shared conditions only (no all-zero Deep Caries target).
# =====================================================================================
import numpy as np, torch, torch.nn as nn, time, os, json

B_SEEDS    = list(SEEDS)
B_BUDGETS  = (16, 32, 64, 128, 256)
B_RANKS    = (2, 4, 8, 16)
B_DEFAULT_BUDGET = 256
B_EPOCHS, B_LR, B_WD, B_LAM_L2 = 40, 5e-4, 1e-4, 1e-2       # same as the original sweep
B_HEADLINE = ["source_only", "R0_thresh", "R1_head", "E_lowrank_r8", "E_linear_l2"]   # other backbones

# ---------- embedding-space correction trainer ----------
def train_correction(Xs_tr, Ys_tr, Xt, Yt, rank=None, mode="none", seed=0):
    torch.manual_seed(seed)
    d = Xt.shape[1]; Zt = torch.tensor(Xt); Yt_ = torch.tensor(Yt[:, TCOLS])
    if mode == "bias":
        v = nn.Parameter(torch.zeros(d)); params = [v]; corr = lambda Z: v.expand_as(Z); trainable = d
    elif rank is None:
        W = nn.Parameter(torch.zeros(d, d)); params = [W]; corr = lambda Z: Z @ W; trainable = d*d
    else:
        A = nn.Parameter(torch.randn(d, rank)*0.02); Bm = nn.Parameter(torch.zeros(rank, d)); params = [A, Bm]
        corr = lambda Z: (Z @ A) @ Bm; trainable = 2*d*rank
    Wr = nn.Parameter(torch.zeros(d, len(TCOLS)))            # readout used only to train the correction
    init = [p.detach().clone() for p in params]
    opt = torch.optim.AdamW(params + [Wr], lr=B_LR if mode != "bias" else 1e-3, weight_decay=B_WD)
    bce = nn.BCEWithLogitsLoss()
    if mode == "replay":
        Zs = torch.tensor(Xs_tr); Ys_ = torch.tensor(Ys_tr[:, TCOLS])
    epochs = B_EPOCHS if mode != "bias" else 60
    for _ in range(epochs):
        opt.zero_grad()
        loss = bce((Zt + corr(Zt)) @ Wr, Yt_)
        if mode == "l2":
            loss = loss + B_LAM_L2 * sum((p - p0).pow(2).sum() for p, p0 in zip(params, init))
        if mode == "replay":
            loss = loss + bce((Zs + corr(Zs)) @ Wr, Ys_)
        loss.backward(); opt.step()
    scale = 0.5 if mode == "avg" else 1.0
    P_ = [p.detach().cpu().numpy().astype(np.float32) for p in params]   # frozen numpy copy
    if mode == "bias":
        corr_np = lambda Z: np.broadcast_to(P_[0][None, :], Z.shape)
    elif rank is None:
        corr_np = lambda Z: Z @ P_[0]
    else:
        corr_np = lambda Z: (Z @ P_[0]) @ P_[1]
    fn = lambda Z: (Z + scale * corr_np(Z)).astype(np.float32)
    Wr_ = Wr.detach().cpu().numpy().astype(np.float32)
    # the target-trained readout, applied exactly as it was trained (full correction, no envelope)
    tgt_head = lambda Z: 1.0 / (1.0 + np.exp(-((Z + corr_np(Z)) @ Wr_)))
    return fn, trainable, tgt_head

def moment_map(X_src_tr, X_tgt_unlab):
    mu_s, sd_s = X_src_tr.mean(0), X_src_tr.std(0) + 1e-6
    mu_t, sd_t = X_tgt_unlab.mean(0), X_tgt_unlab.std(0) + 1e-6
    return lambda Z: ((Z - mu_t) / sd_t * sd_s + mu_s).astype(np.float32)

# ---------- one run ----------
_ENV0 = {}
def source_env(b, seed, Xs, dtr, dva):
    k = (b, seed)
    if k not in _ENV0:
        set_seed(seed); _ENV0[k] = Envelope().fit(Xs[dtr], Y[dtr], Xs[dva], Y[dva])
    return _ENV0[k]

def run_B(b, tech, seed, budget=B_DEFAULT_BUDGET):
    t0 = time.time(); set_seed(seed)
    Xs, Xa = load_dentex_v2(b), load_adcd_v2(b)
    dtr, dva, dte = dentex_split_v2(seed); atr, ava, ate = adcd_split_v2(seed)
    env0 = source_env(b, seed, Xs, dtr, dva)
    sel = label_slice(atr, budget, seed)
    uses_labels = tech not in ("source_only", "R0_thresh", "R0_moment")
    info = {"trainable": 0}
    fn_s = fn_t = (lambda Z: Z)
    tgt_head = None

    if tech == "source_only":
        env = env0
    elif tech == "R0_thresh":
        tau = float(np.quantile(env0._score(Xa[atr]), OOD_Q)); env = with_tau(env0, tau)
    elif tech == "R0_moment":
        fn_t = moment_map(Xs[dtr], Xa[atr]); env = env0
    elif tech == "R1_head":
        g = ADCD_GROUPS[sel]; ug = np.unique(g)
        if len(sel) >= 32 and len(ug) >= 4:
            rng = np.random.RandomState(seed); vg = set(rng.choice(ug, size=max(1, len(ug)//5), replace=False).tolist())
            fit = sel[[x not in vg for x in g]]; val = sel[[x in vg for x in g]]
        else:
            fit = val = sel                      # 16-crop slice: fit and calibrate on the same crops (stated)
        envT = Envelope().fit(Xa[fit], ADCD_Y[fit], Xa[val], ADCD_Y[val])
        env = with_gate(envT, env0)
        info["trainable"] = int(sum(p.numel() for p in envT.head.parameters())) + Xa.shape[1]*len(TCOLS) + 1
        info["fit_n"], info["val_n"] = int(len(fit)), int(len(val))
    else:
        if tech == "E_bias":            fn, tp, tgt_head = train_correction(None, None, Xa[sel], ADCD_Y[sel], mode="bias", seed=seed)
        elif tech.startswith("E_lowrank_r"):
            fn, tp, tgt_head = train_correction(None, None, Xa[sel], ADCD_Y[sel], rank=int(tech.split("_r")[-1]), seed=seed)
        elif tech == "E_linear_l2":     fn, tp, tgt_head = train_correction(None, None, Xa[sel], ADCD_Y[sel], mode="l2", seed=seed)
        elif tech == "E_linear_replay": fn, tp, tgt_head = train_correction(Xs[dtr], Y[dtr], Xa[sel], ADCD_Y[sel], mode="replay", seed=seed)
        elif tech == "E_linear_avg":    fn, tp, tgt_head = train_correction(None, None, Xa[sel], ADCD_Y[sel], mode="avg", seed=seed)
        else: raise ValueError(tech)
        fn_s = fn_t = fn; info["trainable"] = int(tp)
        env = Envelope().fit(fn(Xs[dtr]), Y[dtr], fn(Xs[dva]), Y[dva])

    tgt_gate = fit_target_gate(fn_t(Xa[atr]), fn_t(Xa[ava]))
    fixed = env0._score(Xa[ate])               # unadapted source gate on raw embeddings: one tooth set for all techniques
    tgt, arrays = eval_target_v2(env, fn_t(Xa[ate]), ADCD_Y[ate], ADCD_GROUPS[ate], tgt_gate, fixed_score=fixed)
    src_before = eval_source_v2(env0, Xs[dte], Y[dte])
    src_after = eval_source_v2(env, fn_s(Xs[dte]), Y[dte], tgt_gate)
    # Controls: score the same test crops with (a) the target-trained readout kept as the
    # predictor and (b) the ORIGINAL source envelope applied to the adapted representation, next to the
    # re-fitted source envelope used above. Full coverage and on the crops the main model's gate accepts.
    acc_main = env._score(fn_t(Xa[ate])) <= env.tau
    def _two(P3):
        return {"full": head_block(P3, ADCD_Y[ate]), "src": _block(np.asarray(P3), ADCD_Y[ate][:, TCOLS], acc_main)}
    controls = {"src_env_fixed": _two(env0.proba(fn_t(Xa[ate]))[:, TCOLS])}
    if tgt_head is not None:
        controls["tgt_head"] = _two(tgt_head(Xa[ate]))
    rec = {"cell": "B", "backbone": b, "technique": tech, "seed": seed,
           "budget": int(budget) if uses_labels else 0, "uses_target_labels": uses_labels,
           "uses_unlabeled_target": tech in ("R0_thresh", "R0_moment"), **info,
           "n_test": int(len(ate)), "tgt": tgt, "controls": controls, "src_before": src_before, "src_after": src_after,
           "delta_ok": bool(max(src_after["max_dev"], tgt["max_dev_tgt"]) <= DELTA + 1e-4),
           "seconds": round(time.time()-t0, 1)}
    tag = f"B_{b}_{tech}_n{rec['budget']}_s{seed}"
    save_per_tooth(tag, arrays); rec["per_tooth"] = tag
    ext, ext_arr = eval_ext_v2(env, fn_t, Xa, tgt_gate)
    if ext is not None:
        rec["tgt_ext"] = ext; save_per_tooth(tag + "_ext", ext_arr)
    return rec

def plan_B():
    jobs = []
    for b in V2_BACKBONES:
        if b == "dental_vitl":
            techs = ["source_only", "R0_thresh", "R0_moment", "E_bias", "E_linear_l2", "E_linear_replay", "E_linear_avg"]
            techs += [f"E_lowrank_r{r}" for r in B_RANKS]
            for s in B_SEEDS:
                for t in techs: jobs.append((b, t, s, B_DEFAULT_BUDGET))
                for n in B_BUDGETS:                                   # label-budget curves
                    jobs.append((b, "R1_head", s, n))
                    if n != B_DEFAULT_BUDGET: jobs.append((b, "E_lowrank_r8", s, n))
        else:
            for s in B_SEEDS:
                for t in B_HEADLINE: jobs.append((b, t, s, B_DEFAULT_BUDGET))
    return jobs

B_RUN = globals().get("B_RUN", True)    # set False to define the functions only and keep saved results
B_ROWS = []
JOBS = plan_B() if B_RUN else []; print(f"Cell B: {len(JOBS)} distinct runs" if B_RUN else "Cell B: functions defined; saved results kept")
for i, (b, t, s, n) in enumerate(JOBS):
    try:
        r = run_B(b, t, s, n); B_ROWS.append(r)
        print(f"[{i+1}/{len(JOBS)}] {b:11s} {t:16s} n={r['budget']:3d} s{s} | full {r['tgt']['full']['macro']:.3f} "
              f"| src-gate {r['tgt']['src']['macro']:.3f} @cov {r['tgt']['src']['coverage']:.2f} "
              f"| tgt-gate cov {r['tgt']['tgt']['coverage']:.2f} | DENTEX {r['src_before']['macro4_full']:.3f}->{r['src_after']['macro4_full']:.3f} "
              f"| delta_ok {r['delta_ok']} [{r['seconds']}s]")
    except Exception as e:
        import traceback; traceback.print_exc()
        B_ROWS.append({"cell": "B", "backbone": b, "technique": t, "seed": s, "budget": n, "error": repr(e)})
    if (i+1) % 10 == 0: save_rows(B_ROWS, "B_frozen_ladder")
if B_RUN:
    save_rows(B_ROWS, "B_frozen_ladder")
    print("saved v2/B_frozen_ladder.json/.csv and per-tooth scores in v2/per_tooth/")

## 8b. Stage B2: two-backbone fusion (re-run with corrected labels)
The paper's fusion result used the old image-level ADCD labels, so it is repeated here on the V2 embeddings.

In [ ]:
# =====================================================================================
# STAGE B2 - two-backbone fusion on the tooth-context embeddings
# Every fused and single model is also scored on the held-out ADCD external set.
# Each backbone is first adapted with the rank-8 embedding-space low-rank correction (256 labeled
# ADCD train crops), as in the original fusion experiment. Then:
#   concat : one envelope on the concatenated adapted embeddings
#   late   : per-backbone envelopes; probabilities averaged; a tooth is accepted only if every
#            backbone's gate accepts it
# Rows also include each single adapted backbone on the same split, for a like-for-like comparison.
# =====================================================================================
assert "train_correction" in globals(), "run Stage B first (it defines train_correction)"
F_BACKBONES = [b for b in ("dental_vitl", "dinov3") if b in V2_BACKBONES]
F_BUDGET, F_RANK = 256, 8
F_ROWS = []
if len(F_BACKBONES) < 2:
    print("fusion needs both dental_vitl and dinov3 embeddings; skipped")
else:
    for s in SEEDS:
        set_seed(s)
        dtr, dva, dte = dentex_split_v2(s); atr, ava, ate = adcd_split_v2(s)
        sel = label_slice(atr, F_BUDGET, s)
        XS, XA, ENV = {}, {}, {}
        for b in F_BACKBONES:
            Xs, Xa = load_dentex_v2(b), load_adcd_v2(b)
            fn, _, _ = train_correction(None, None, Xa[sel], ADCD_Y[sel], rank=F_RANK, seed=s)
            XS[b], XA[b] = fn(Xs), fn(Xa)
            set_seed(s); ENV[b] = Envelope().fit(XS[b][dtr], Y[dtr], XS[b][dva], Y[dva])
            tg = fit_target_gate(XA[b][atr], XA[b][ava])
            res, arr = eval_target_v2(ENV[b], XA[b][ate], ADCD_Y[ate], ADCD_GROUPS[ate], tg)
            tag = f"F_single_{b}_s{s}"; save_per_tooth(tag, arr)
            ext, ext_arr = eval_ext_v2(ENV[b], lambda Z: Z, XA[b], tg)
            F_ROWS.append({"cell": "F", "backbone": "fusion", "technique": f"single_{b}", "seed": s, "budget": F_BUDGET,
                           "tgt": res, **({"tgt_ext": ext} if ext else {}), "dentex": eval_source_v2(ENV[b], XS[b][dte], Y[dte]), "per_tooth": tag})
            if ext: save_per_tooth(tag + "_ext", ext_arr)
        Xc_s = np.concatenate([XS[b] for b in F_BACKBONES], 1); Xc_a = np.concatenate([XA[b] for b in F_BACKBONES], 1)
        set_seed(s); envC = Envelope().fit(Xc_s[dtr], Y[dtr], Xc_s[dva], Y[dva])
        tg = fit_target_gate(Xc_a[atr], Xc_a[ava])
        res, arr = eval_target_v2(envC, Xc_a[ate], ADCD_Y[ate], ADCD_GROUPS[ate], tg)
        tag = f"F_concat_s{s}"; save_per_tooth(tag, arr)
        ext, ext_arr = eval_ext_v2(envC, lambda Z: Z, Xc_a, tg)
        F_ROWS.append({"cell": "F", "backbone": "fusion", "technique": "concat", "seed": s, "budget": F_BUDGET,
                       "tgt": res, **({"tgt_ext": ext} if ext else {}), "dentex": eval_source_v2(envC, Xc_s[dte], Y[dte]), "per_tooth": tag})
        if ext: save_per_tooth(tag + "_ext", ext_arr)
        # late fusion: average probabilities, accept only if all gates accept
        P = np.mean([ENV[b].proba(XA[b][ate])[:, TCOLS] for b in F_BACKBONES], 0); Yt = ADCD_Y[ate][:, TCOLS]
        ratio = np.max([ENV[b]._score(XA[b][ate]) / ENV[b].tau for b in F_BACKBONES], 0)   # <= 1 iff every gate accepts
        acc = ratio <= 1.0
        res = {"full": _block(P, Yt, np.ones(len(Yt), bool)), "src": _block(P, Yt, acc)}
        order = np.argsort(ratio)
        for f in MATCHED_COVERAGE:
            mk = np.zeros(len(Yt), bool); mk[order[:max(1, int(round(f*len(Yt))))]] = True; res[f"cov{int(f*100)}"] = _block(P, Yt, mk)
        tag = f"F_late_s{s}"
        save_per_tooth(tag, {"probs": P.astype(np.float32), "labels": Yt.astype(np.int8), "groups": ADCD_GROUPS[ate],
                             "maha_src": ratio.astype(np.float32), "tau_src": np.float32(1.0)})
        late_row = {"cell": "F", "backbone": "fusion", "technique": "late", "seed": s, "budget": F_BUDGET, "tgt": res, "per_tooth": tag}
        if HAS_EXT:
            ei = ADCD_EXT_IDX
            Pe = np.mean([ENV[b].proba(XA[b][ei])[:, TCOLS] for b in F_BACKBONES], 0); Ye = ADCD_Y[ei][:, TCOLS]
            re_ = np.max([ENV[b]._score(XA[b][ei]) / ENV[b].tau for b in F_BACKBONES], 0)
            late_row["tgt_ext"] = {"full": _block(Pe, Ye, np.ones(len(Ye), bool)), "src": _block(Pe, Ye, re_ <= 1.0)}
            save_per_tooth(tag + "_ext", {"probs": Pe.astype(np.float32), "labels": Ye.astype(np.int8), "groups": ADCD_GROUPS[ei],
                                          "maha_src": re_.astype(np.float32), "tau_src": np.float32(1.0)})
        F_ROWS.append(late_row)
        print(f"s{s}: full macro  " + "  ".join(f"{r['technique']} {r['tgt']['full']['macro']:.3f}" for r in F_ROWS if r["seed"] == s)
              + "  | caries full " + "  ".join(f"{r['technique']} {r['tgt']['full']['auc_Caries']:.3f}" for r in F_ROWS if r["seed"] == s))
    save_rows(F_ROWS, "B2_fusion"); print("saved B2_fusion.json/.csv")

## 9. Stage C: live-trunk PEFT and full-trunk fine-tuning (three seeds)
Longest stage (several hours). Resumable: stop and re-run the cell, finished runs are skipped. Edit `C_BACKBONES`, `C_METHODS`, `C_SEEDS` to control compute.

In [ ]:
# =====================================================================================
# STAGE C - live-trunk PEFT (5 methods) and full-trunk fine-tuning (3), three seeds
# -------------------------------------------------------------------------------------
# Needs: Stage A (or section 7 after a restart).
#
# Design:
#   * per-box ADCD labels and 15% crop margin (Cell A), same as DENTEX
#   * labeled slice drawn from ADCD TRAIN only; target scored on ADCD TEST only
#   * target loss on the three shared conditions only
#   * three seeds; per-tooth scores saved for cluster bootstrap
#   * source scored on DENTEX TEST
#   * EWC: head first fitted on DENTEX (trunk frozen), then Fisher and anchor taken at that source model
#   * rehearsal: one source batch after every target batch (interleaved 1:1)
#   * controls: target-trained head and original source envelope scored on the same test crops
#   * full_featanchor: feature-anchoring penalty to a frozen copy of the trunk
#   * resumable: finished (backbone, method, seed) runs found in v2/C_live_trunk.json
#     are skipped, so the cell can be stopped and restarted
#
# Full-trunk methods (EWC, rehearsal, feature anchoring) all start from the same source-trained head.
#
# Time on the earlier single-seed pass: about 11 min per Dental ViT-L PEFT run and
# 3 min per DINOv3 PEFT run. C_TGT_GATE=True adds an embedding pass over ADCD train.
# =====================================================================================
import torch, torch.nn as nn, numpy as np, os, json, time, copy, math
from torch.utils.data import Dataset, DataLoader
from PIL import Image

C_BACKBONES = ["dental_vitl", "dinov3"]
C_METHODS   = ["lora", "adapter", "vpt", "bitfit", "ln_affine",          # live-trunk PEFT
               "full_ewc", "full_rehearsal", "full_featanchor"]           # full-trunk (~300M)
C_SEEDS     = [42, 43, 44]
C_BUDGET    = 256
C_TGT_GATE  = True
SAVE_TRUNK_CKPT = False        # ~1 GB per full-trunk run if True

PEFT_EPOCHS, PEFT_BS, PEFT_LR = 3, 8, 1e-4
FT_EPOCHS, FT_BS, FT_LR       = 3, 8, 1e-5
EWC_LAMBDA, FEAT_ANCHOR_W     = 1e3, 0.1
EWC_HEAD_EPOCHS               = 1         # source-head fit before Fisher estimation (trunk frozen)
LORA_RANK, LORA_ALPHA, ADAPTER_DIM, VPT_NTOKENS = 8, 16, 32, 10

# Never reuse an earlier Stage C result unless all run-defining inputs match.
import hashlib
_C_SIGNATURE_DATA = {
    "version": "Paper2_Rerun_V5", "run_folder": V2_NAME,
    # The lists of backbones, methods and seeds are NOT part of the signature: they select which runs to do,
    # they do not change any individual run. This keeps resume working if a list is shortened or extended.
    "budget": C_BUDGET, "target_gate": C_TGT_GATE,
    "peft": [PEFT_EPOCHS, PEFT_BS, PEFT_LR, LORA_RANK, LORA_ALPHA, ADAPTER_DIM, VPT_NTOKENS],
    "full": [FT_EPOCHS, FT_BS, FT_LR, EWC_LAMBDA, EWC_HEAD_EPOCHS, FEAT_ANCHOR_W],
    "crop_mode": globals().get("ADCD_CROP_MODE", "from_saved_config"), "pad": PAD,
    "harmonize_caries": HARMONIZE_CARIES,
    "label_sha256": hashlib.sha256(ADCD_Y.tobytes() + Y.tobytes()).hexdigest(),
    "group_sha256": hashlib.sha256(ADCD_GROUPS.tobytes() + DENTEX_GROUPS.tobytes()).hexdigest(),
}
C_RUN_SIGNATURE = hashlib.sha256(json.dumps(_C_SIGNATURE_DATA, sort_keys=True).encode()).hexdigest()
print("Stage C run signature:", C_RUN_SIGNATURE[:16])


# ---------- PEFT modules (unchanged) ----------
class LoRALinear(nn.Module):
    def __init__(self, lin, r=8, alpha=16):
        super().__init__(); self.lin = lin
        for p in self.lin.parameters(): p.requires_grad = False
        self.A = nn.Parameter(torch.zeros(r, lin.in_features)); self.B = nn.Parameter(torch.zeros(lin.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5)); self.s = alpha / r
    def forward(self, x): return self.lin(x) + (x @ self.A.t() @ self.B.t()) * self.s

class Adapter(nn.Module):
    def __init__(self, d, h=32):
        super().__init__(); self.down = nn.Linear(d, h); self.act = nn.GELU(); self.up = nn.Linear(h, d)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x): return x + self.up(self.act(self.down(x)))

def _iter_linears(module):
    for name, m in module.named_modules():
        if isinstance(m, nn.Linear): yield name, m

def _set_by_name(root, dotted, newmod):
    parts = dotted.split("."); obj = root
    for p in parts[:-1]: obj = getattr(obj, p) if not p.isdigit() else obj[int(p)]
    setattr(obj, parts[-1], newmod) if not parts[-1].isdigit() else obj.__setitem__(int(parts[-1]), newmod)

def apply_peft(trunk, method):
    for p in trunk.parameters(): p.requires_grad = False
    nt, status = 0, "ok"
    if method == "bitfit":
        for n, p in trunk.named_parameters():
            if n.endswith("bias"): p.requires_grad = True; nt += p.numel()
    elif method == "ln_affine":
        for n, p in trunk.named_parameters():
            low = n.lower()
            if ("norm" in low or "ln" in low) and (n.endswith("weight") or n.endswith("bias")):
                p.requires_grad = True; nt += p.numel()
    elif method == "lora":
        targets = [(n, m) for n, m in _iter_linears(trunk) if any(k in n.lower() for k in ["query", "value", "q_proj", "v_proj", "qkv"])]
        if not targets: targets = [(n, m) for n, m in _iter_linears(trunk) if "attn" in n.lower() or "attention" in n.lower()]
        for n, m in targets:
            l = LoRALinear(m, r=LORA_RANK, alpha=LORA_ALPHA); _set_by_name(trunk, n, l); nt += l.A.numel() + l.B.numel()
    elif method == "adapter":
        seen = set()
        for n, m in list(_iter_linears(trunk)):
            if not any(k in n.lower() for k in ["fc2", "proj", "dense", "out"]): continue
            parent = n.rsplit(".", 1)[0]
            if parent in seen: continue
            seen.add(parent); ad = Adapter(m.out_features, ADAPTER_DIM)
            _set_by_name(trunk, n, nn.Sequential(m, ad)); nt += sum(p.numel() for p in ad.parameters())
    elif method == "vpt":
        status = "vpt_in_wrapper"
    if nt == 0 and method not in ("vpt",): status = f"unsupported: no parameters found for {method}"
    return nt, status

# ---------- wrappers (unchanged logic) ----------
class DentalNet(nn.Module):
    def __init__(self, vit, k, vpt=0):
        super().__init__(); self.vit = vit; self.head = nn.Linear(1024, k); self.vpt = vpt
        if vpt > 0: self.prompt = nn.Parameter(torch.randn(1, vpt, 1024) * 0.02)
    def forward(self, x):
        emb = self.vit.transformer.embeddings(x)
        if self.vpt > 0:
            B = emb.shape[0]; emb = torch.cat([emb[:, :1], self.prompt.expand(B, -1, -1), emb[:, 1:]], dim=1)
        enc, _ = self.vit.transformer.encoder(emb); cls = enc[:, 0]
        return self.head(cls), cls

class DinoNet(nn.Module):
    def __init__(self, model, k, vpt=0):
        super().__init__(); self.model = model; self.head = nn.Linear(1024, k); self.vpt = vpt
        if vpt > 0:
            self.prompt = nn.Parameter(torch.randn(1, vpt, 1024) * 0.02)
            emb_mod = getattr(self.model, "embeddings", None)
            if emb_mod is None: raise RuntimeError("DINOv3 embeddings module not found for VPT")
            def _append(module, inp, out):
                seq = out[0] if isinstance(out, tuple) else out; B = seq.shape[0]
                seq = torch.cat([seq[:, :1], self.prompt.expand(B, -1, -1), seq[:, 1:]], dim=1)
                return (seq,) + tuple(out[1:]) if isinstance(out, tuple) else seq
            emb_mod.register_forward_hook(_append)
    def forward(self, pixel_values):
        o = self.model(pixel_values=pixel_values); f = getattr(o, "pooler_output", None)
        if f is None: f = o.last_hidden_state[:, 0]
        return self.head(f), f

def load_dental_vitl_trainable():
    import sys, subprocess, torchvision.transforms as T
    from huggingface_hub import hf_hub_download
    if not os.path.isdir("ViT-pytorch"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jeonsworld/ViT-pytorch"], check=True)
    if "ViT-pytorch" not in sys.path: sys.path.insert(0, "ViT-pytorch")
    from models.modeling import VisionTransformer, CONFIGS
    ck = hf_hub_download(repo_id="Bryceee/Teeth_Visual_Experts_Models",
                         filename="Teeth_Visual_Experts_ViT_L_periapical_images_cls_7diseases.bin")
    state = torch.load(ck, map_location="cpu")
    if isinstance(state, dict) and "state_dict" in state: state = state["state_dict"]
    pos = state["transformer.embeddings.position_embeddings"].shape[1]
    img = int(round(math.sqrt(pos - 1))) * 16; nH = state["head.weight"].shape[0]
    m = VisionTransformer(CONFIGS["ViT-L_16"], img_size=img, num_classes=nH, zero_head=False)
    m.load_state_dict(state, strict=False)
    pre = T.Compose([T.Resize((img, img)), T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)])
    return m, pre

# ---------- data ----------
class CropDSv2(Dataset):
    """kind 'dentex': DENTEX tooth bbox [x,y,w,h]; kind 'adcd': ADCD box [x,y,w,h]. Both use crop_tooth
    with the same 15% margin."""
    def __init__(self, recs, pre=None, dino_proc=None):
        self.recs, self.pre, self.dino = recs, pre, dino_proc; self._last = (None, None)
    def __len__(self): return len(self.recs)
    def _img(self, kind, f):
        if self._last[0] != f:
            self._last = (f, load_image(f) if kind == "dentex" else Image.open(f).convert("RGB"))
        return self._last[1]
    def __getitem__(self, i):
        f, y, box, kind = self.recs[i]
        c = crop_tooth(self._img(kind, f), box, pad=PAD).convert("RGB")
        px = self.dino(images=[c], return_tensors="pt")["pixel_values"][0] if self.dino is not None else self.pre(c)
        return px, torch.tensor(y, dtype=torch.float32)

def recs_dentex(idx): return [(teeth[i]["file"], teeth[i]["y"], teeth[i]["bbox"], "dentex") for i in idx]
def recs_adcd(idx):   return [(ADCD_RECS[i]["path"], ADCD_Y[i], ADCD_RECS[i].get("xywh_crop", ADCD_RECS[i]["xywh"]), "adcd") for i in idx]

def compute_fisher(model, loader, n_batches=20):
    fisher = {n: torch.zeros_like(p) for n, p in model.named_parameters() if p.requires_grad}
    model.eval(); crit = nn.BCEWithLogitsLoss(); seen = 0
    for bi, (x, y) in enumerate(loader):
        if bi >= n_batches: break
        model.zero_grad(); logits, _ = model(x.to(DEVICE)); crit(logits, y.to(DEVICE)).backward()
        for n, p in model.named_parameters():
            if p.requires_grad and p.grad is not None: fisher[n] += p.grad.detach() ** 2
        seen += 1
    for n in fisher: fisher[n] /= max(1, seen)
    return fisher

def ewc_penalty(model, fisher, star):
    return sum((fisher[n] * (p - star[n]) ** 2).sum() for n, p in model.named_parameters() if n in fisher)

# ---------- one run ----------
_C_ENV0 = {}
def run_C(b, method, seed):
    t0 = time.time(); set_seed(seed); k = NUM_DX
    is_ft = method.startswith("full_")
    vpt = VPT_NTOKENS if method == "vpt" else 0
    if b == "dental_vitl":
        base, pre = load_dental_vitl_trainable(); model = DentalNet(base, k, vpt); proc = None
    else:
        base, proc = load_dinov3(); model = DinoNet(base, k, vpt); pre = None
    trunk = model.vit if b == "dental_vitl" else model.model
    if is_ft:
        for p in model.parameters(): p.requires_grad = True
        n_adapt, status = sum(p.numel() for p in trunk.parameters()), "ok"
    else:
        n_adapt, status = apply_peft(trunk, method)
        for p in model.head.parameters(): p.requires_grad = True
        if vpt: model.prompt.requires_grad = True; n_adapt += model.prompt.numel()
    if status.startswith("unsupported"):
        return {"cell": "C", "backbone": b, "method": method, "seed": seed, "error": status}
    model = model.to(DEVICE)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

    dtr, dva, dte = dentex_split_v2(seed); atr, ava, ate = adcd_split_v2(seed)
    sel = label_slice(atr, C_BUDGET, seed)
    tgt_dl = DataLoader(CropDSv2(recs_adcd(sel), pre, proc), batch_size=FT_BS if is_ft else PEFT_BS, shuffle=True)
    src_dl = DataLoader(CropDSv2(recs_dentex(dtr), pre, proc), batch_size=FT_BS, shuffle=True) if is_ft else None
    crit = nn.BCEWithLogitsLoss(); tcols = torch.tensor(TCOLS, device=DEVICE)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=FT_LR if is_ft else PEFT_LR, weight_decay=1e-4)
    star = fisher = frozen = None
    head_fit_loss = []
    if is_ft:
        # All three full-trunk methods start from the SAME source-trained state for a given backbone and seed:
        # the classification head is first fitted on DENTEX train with the trunk frozen (EWC_HEAD_EPOCHS epochs).
        # EWC then takes its Fisher information and anchor at this source model; rehearsal and feature anchoring
        # start from it as well.
        req = {n: p.requires_grad for n, p in model.named_parameters()}
        for n, p in model.named_parameters(): p.requires_grad = n.startswith("head.")
        hopt = torch.optim.AdamW(model.head.parameters(), lr=1e-3, weight_decay=1e-4); model.train()
        for _ in range(EWC_HEAD_EPOCHS):
            tot, nb = 0.0, 0
            for x, y in src_dl:
                hopt.zero_grad(); lg, _f = model(x.to(DEVICE)); l_ = crit(lg, y.to(DEVICE)); l_.backward(); hopt.step()
                tot += float(l_.detach()); nb += 1
            head_fit_loss.append(round(tot / max(1, nb), 4))
        for n, p in model.named_parameters(): p.requires_grad = req[n]
        if method == "full_ewc":
            star = {n: p.detach().clone() for n, p in model.named_parameters()}; fisher = compute_fisher(model, src_dl)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=FT_LR, weight_decay=1e-4)
    if method == "full_featanchor":
        frozen = copy.deepcopy(model).eval()
        for p in frozen.parameters(): p.requires_grad = False

    def _step(x, y, kind):
        x, y = x.to(DEVICE), y.to(DEVICE); opt.zero_grad()
        logits, feat = model(x)
        loss = crit(logits.index_select(1, tcols), y.index_select(1, tcols)) if kind == "tgt" else crit(logits, y)
        if method == "full_ewc": loss = loss + (EWC_LAMBDA / 2.0) * ewc_penalty(model, fisher, star)
        if method == "full_featanchor":
            with torch.no_grad(): _, f0 = frozen(x)
            loss = loss + FEAT_ANCHOR_W * ((feat - f0) ** 2).mean()
        loss.backward(); opt.step()
    n_updates = {"tgt": 0, "src": 0}
    src_iter = iter(src_dl) if method == "full_rehearsal" else None
    model.train()
    for ep in range(FT_EPOCHS if is_ft else PEFT_EPOCHS):
        for x, y in tgt_dl:
            _step(x, y, "tgt"); n_updates["tgt"] += 1
            if src_iter is not None:          # rehearsal: one source batch after every target batch (1:1, interleaved)
                try: xs, ys = next(src_iter)
                except StopIteration: src_iter = iter(src_dl); xs, ys = next(src_iter)
                _step(xs, ys, "src"); n_updates["src"] += 1
        print(f"   [{b}/{method}/s{seed}] epoch {ep+1} ({time.time()-t0:.0f}s)")
    train_min = (time.time() - t0) / 60

    model.eval()
    HEAD_P = {}
    def embed(recs, keep_head=None):
        dl = DataLoader(CropDSv2(recs, pre, proc), batch_size=16); Z = []; H = []
        with torch.no_grad():
            for x, _ in dl:
                lg, f = model(x.to(DEVICE)); Z.append(torch.nn.functional.normalize(f.float().cpu(), dim=1).numpy())
                if keep_head: H.append(torch.sigmoid(lg.float().cpu()).numpy())
        if keep_head: HEAD_P[keep_head] = np.concatenate(H, 0)
        return np.concatenate(Z, 0).astype(np.float32)
    Zs_tr, Zs_va, Zs_te = embed(recs_dentex(dtr)), embed(recs_dentex(dva)), embed(recs_dentex(dte))
    Za_te, Za_va = embed(recs_adcd(ate), keep_head="ate"), embed(recs_adcd(ava))
    tgt_gate = fit_target_gate(embed(recs_adcd(atr)), Za_va) if C_TGT_GATE else None
    env = Envelope().fit(Zs_tr, Y[dtr], Zs_va, Y[dva])
    if (b, seed) not in _C_ENV0:                      # frozen-trunk reference from Stage A embeddings
        Xs0 = load_dentex_v2(b); _C_ENV0[(b, seed)] = (Envelope().fit(Xs0[dtr], Y[dtr], Xs0[dva], Y[dva]), Xs0)
    env0, Xs0 = _C_ENV0[(b, seed)]
    fixed = env0._score(load_adcd_v2(b)[ate])        # unadapted source gate: one tooth set shared by all techniques
    tgt, arrays = eval_target_v2(env, Za_te, ADCD_Y[ate], ADCD_GROUPS[ate], tgt_gate, fixed_score=fixed)
    acc_main = env._score(Za_te) <= env.tau
    def _two(P3):
        return {"full": head_block(P3, ADCD_Y[ate]), "src": _block(np.asarray(P3), ADCD_Y[ate][:, TCOLS], acc_main)}
    controls = {"tgt_head": _two(HEAD_P["ate"][:, TCOLS]),                 # head trained with the adaptation, kept
                "src_env_fixed": _two(env0.proba(Za_te)[:, TCOLS])}        # original source envelope on adapted features
    rec = {"cell": "C", "backbone": b, "method": method, "seed": seed, "budget": C_BUDGET,
           "kind": "full_trunk" if is_ft else "live_peft", "adapt_params": int(n_adapt), "trainable_incl_head": int(trainable),
           "status": status, "n_test": int(len(ate)), "tgt": tgt, "controls": controls, "optimizer_updates": n_updates,
           "source_head_fit_loss_per_epoch": head_fit_loss, "readout_params": int(sum(p.numel() for p in model.head.parameters())),
           "src_before": eval_source_v2(env0, Xs0[dte], Y[dte]), "src_after": eval_source_v2(env, Zs_te, Y[dte], tgt_gate),
           "train_minutes": round(train_min, 1), "total_minutes": round((time.time()-t0)/60, 1), "device": str(DEVICE), "run_signature": C_RUN_SIGNATURE}
    rec["delta_ok"] = bool(max(rec["src_after"]["max_dev"], tgt["max_dev_tgt"]) <= DELTA + 1e-4)
    tag = f"C_{b}_{method}_s{seed}"; save_per_tooth(tag, arrays); rec["per_tooth"] = tag
    if HAS_EXT:
        Za_ext = embed(recs_adcd(ADCD_EXT_IDX))
        ext, ext_arr = eval_target_v2(env, Za_ext, ADCD_Y[ADCD_EXT_IDX], ADCD_GROUPS[ADCD_EXT_IDX], tgt_gate)
        rec["tgt_ext"] = ext; save_per_tooth(tag + "_ext", ext_arr)
    if SAVE_TRUNK_CKPT:
        ck = os.path.join(V2, "checkpoints"); os.makedirs(ck, exist_ok=True)
        torch.save({"state_dict": model.state_dict(), "meta": {k_: v for k_, v in rec.items() if k_ != "tgt"}}, os.path.join(ck, f"{tag}.pt"))
    del model, frozen
    if DEVICE == "mps":
        try: rec["mps_driver_alloc_gb"] = round(torch.mps.driver_allocated_memory() / 1e9, 2)
        except Exception: pass
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        rec["cuda_peak_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2); torch.cuda.reset_peak_memory_stats()
    return rec

C_PATH = os.path.join(V2, "C_live_trunk.json")
C_ROWS = json.load(open(C_PATH)) if os.path.exists(C_PATH) else []
C_REDO = globals().get("C_REDO", [])     # methods to recompute even if finished
done = {(r["backbone"], r["method"], r["seed"]) for r in C_ROWS
        if "error" not in r and r["method"] not in C_REDO
        and r.get("run_signature") == C_RUN_SIGNATURE
        and os.path.isfile(os.path.join(PT_DIR, r.get("per_tooth", "") + ".npz"))
        and (not HAS_EXT or ("tgt_ext" in r and os.path.isfile(os.path.join(PT_DIR, r.get("per_tooth", "") + "_ext.npz"))))
        and not (r["method"].startswith("full_") and not r.get("source_head_fit_loss_per_epoch"))}
print(f"Stage C resume: {len(done)} verified runs; other saved entries will be re-run")
for b in C_BACKBONES:
    if b not in V2_BACKBONES: print(f"skip {b}: not embedded in Cell A"); continue
    for s in C_SEEDS:
        for m in C_METHODS:
            if (b, m, s) in done: continue
            try:
                r = run_C(b, m, s)
            except Exception as e:
                import traceback; traceback.print_exc(); r = {"cell": "C", "backbone": b, "method": m, "seed": s, "error": repr(e)}
            C_ROWS = [x for x in C_ROWS if not (x.get("backbone") == b and x.get("method") == m and x.get("seed") == s)] + [r]
            save_rows(C_ROWS, "C_live_trunk")
            if "error" not in r:
                print(f"DONE {b}/{m}/s{s}: full {r['tgt']['full']['macro']:.3f} | src-gate {r['tgt']['src']['macro']:.3f} "
                      f"@cov {r['tgt']['src']['coverage']:.2f} | DENTEX {r['src_after']['macro4_full']:.3f} "
                      f"| delta_ok {r['delta_ok']} [{r['total_minutes']} min]")
print("saved v2/C_live_trunk.json/.csv")

## 10. Stage D: joint two-center training at matched budgets, and the transferability test

In [ ]:
# =====================================================================================
# STAGE D - joint two-center training at matched label budgets, and the
#             condition-wise transferability test (CTT) on a separate partition
# -------------------------------------------------------------------------------------
# Needs: Stage A (or section 7), and Stage B finished. The CTT outcome check reads
# B_frozen_ladder.json and C_live_trunk.json if present.
#
# Part 1: source-only, target-only and joint envelopes, with the
#   target label budget matched to the ladder (16, 64, 256, all ADCD train), scored on
#   the ADCD test split at full coverage and under each regime's own gate.
#
# Part 2: CTT estimated WITHOUT touching ADCD test.
#   * ADCD train is split by image into dev_fit (70%) and dev_eval (30%).
#   * A_S->T : source envelope scored on dev_eval;  A_T : envelope fitted on n labeled
#     dev_fit crops (n in CTT_BUDGETS, "all" = whole dev_fit), scored on dev_eval.
#   * Both are FULL-COVERAGE AUROCs on the same dev_eval crops (no gate), so they are
#     computed on a common population.
#   * G = A_T - A_S->T, H = A_T - 0.5, R = G/H. If H <= 0 the ratio is not computed and
#     the condition is reported as "unlearnable". A negative G gives R < 0 (transferable).
#   * Uncertainty: image-cluster bootstrap of R on dev_eval (both AUROCs recomputed on
#     each resample), giving a 95% interval and P(R >= rho).
#   * Outcome, measured on the untouched ADCD test split with ONE adaptation protocol fixed before the
#     run (CTT_OUTCOME_PROTOCOL: rank-8 embedding-space low-rank correction, 256 labeled crops):
#     R_out = (A_T,test - A_protocol,test) / (A_T,test - 0.5); "non-transferable" when R_out >= rho.
#     The best-of-all-techniques outcome is also stored, labeled optimistic (selected on test).
#
# Part 3: crop-geometry-only baseline. A logistic regression on box size,
#   aspect ratio, area and position (normalized by image size), per condition. If geometry alone
#   predicts the condition, crop construction leaks class information.
#     The outcome is also computed against a target-only reference trained with the same 256-label budget.
#   * Reverse direction ADCD -> DENTEX on the three shared conditions, same procedure.
#   * Part 1 also has a domain-balanced joint control (target crops resampled to the DENTEX count).
# =====================================================================================
import numpy as np, os, json, time
from collections import defaultdict

D_BACKBONES   = list(V2_BACKBONES)
D_SEEDS       = list(SEEDS)
JOINT_BUDGETS = (16, 64, 256, "all")
CTT_BUDGETS   = (64, 256, "all")
RHO           = 0.5
CTT_OUTCOME_PROTOCOL = ("E_lowrank_r8", 256)      # fixed before the run; do not change after seeing results
N_BOOT        = 1000

def _fit(X, Yl, idx_fit, idx_val, seed):
    set_seed(seed); return Envelope().fit(X[idx_fit], Yl[idx_fit], X[idx_val], Yl[idx_val])

def _split_idx_by_group(idx, groups, frac, seed):
    g = groups[idx]; ug = np.unique(g); rng = np.random.RandomState(seed); rng.shuffle(ug)
    hold = set(ug[:max(1, int(round(frac*len(ug))))].tolist())
    m = np.array([x in hold for x in g]); return idx[~m], idx[m]

def _budget_idx(idx, n, seed):
    if n == "all" or n >= len(idx): return idx
    return idx[np.random.RandomState(seed).permutation(len(idx))[:n]]

# ---------------- Part 1: joint training at matched budgets ----------------
D1_ROWS = []
for b in D_BACKBONES:
    Xs, Xa = load_dentex_v2(b), load_adcd_v2(b)
    for s in D_SEEDS:
        dtr, dva, dte = dentex_split_v2(s); atr, ava, ate = adcd_split_v2(s)
        envS = _fit(Xs, Y, dtr, dva, s)
        base = {"backbone": b, "seed": s}
        resS, arrS = eval_target_v2(envS, Xa[ate], ADCD_Y[ate], ADCD_GROUPS[ate])
        extS, extA = eval_ext_v2(envS, lambda Z: Z, Xa)
        D1_ROWS.append({**base, "regime": "source_only", "budget": 0, "tgt": resS, **({"tgt_ext": extS} if extS else {}),
                        "dentex": eval_source_v2(envS, Xs[dte], Y[dte])})
        save_per_tooth(f"D_{b}_source_only_n0_s{s}", arrS)
        if extS: save_per_tooth(f"D_{b}_source_only_n0_s{s}_ext", extA)
        for n in JOINT_BUDGETS:
            sel = _budget_idx(atr, n, s) if n == "all" else label_slice(atr, n, s)
            fit, val = (sel, ava) if n == "all" else _split_idx_by_group(sel, ADCD_GROUPS, 0.2, s) if len(sel) >= 32 else (sel, sel)
            envT = _fit(Xa, ADCD_Y, fit, val, s)
            resT, arrT = eval_target_v2(envT, Xa[ate], ADCD_Y[ate], ADCD_GROUPS[ate])
            extT, extTA = eval_ext_v2(envT, lambda Z: Z, Xa)
            D1_ROWS.append({**base, "regime": "target_only", "budget": n if n == "all" else int(n), "n_fit": int(len(fit)),
                            "tgt": resT, **({"tgt_ext": extT} if extT else {}), "dentex": eval_source_v2(envT, Xs[dte], Y[dte])})
            save_per_tooth(f"D_{b}_target_only_n{n}_s{s}", arrT)
            if extT: save_per_tooth(f"D_{b}_target_only_n{n}_s{s}_ext", extTA)
            Xj = np.concatenate([Xs[dtr], Xa[fit]]); Yj = np.concatenate([Y[dtr], ADCD_Y[fit]])
            Xjv = np.concatenate([Xs[dva], Xa[val]]); Yjv = np.concatenate([Y[dva], ADCD_Y[val]])
            set_seed(s); envJ = Envelope().fit(Xj, Yj, Xjv, Yjv)
            resJ, arrJ = eval_target_v2(envJ, Xa[ate], ADCD_Y[ate], ADCD_GROUPS[ate])
            extJ, extJA = eval_ext_v2(envJ, lambda Z: Z, Xa)
            D1_ROWS.append({**base, "regime": "joint", "budget": n if n == "all" else int(n), "n_fit": int(len(fit)),
                            "tgt": resJ, **({"tgt_ext": extJ} if extJ else {}), "dentex": eval_source_v2(envJ, Xs[dte], Y[dte])})
            save_per_tooth(f"D_{b}_joint_n{n}_s{s}", arrJ)
            if extJ: save_per_tooth(f"D_{b}_joint_n{n}_s{s}_ext", extJA)
            # domain-balanced joint control: target crops resampled with replacement to match the DENTEX count
            rngb = np.random.RandomState(s)
            fit_b = fit[rngb.randint(0, len(fit), len(dtr))]; val_b = val[rngb.randint(0, len(val), len(dva))]
            Xjb = np.concatenate([Xs[dtr], Xa[fit_b]]); Yjb = np.concatenate([Y[dtr], ADCD_Y[fit_b]])
            Xjbv = np.concatenate([Xs[dva], Xa[val_b]]); Yjbv = np.concatenate([Y[dva], ADCD_Y[val_b]])
            set_seed(s); envJB = Envelope().fit(Xjb, Yjb, Xjbv, Yjbv)
            resJB, arrJB = eval_target_v2(envJB, Xa[ate], ADCD_Y[ate], ADCD_GROUPS[ate])
            extJB, extJBA = eval_ext_v2(envJB, lambda Z: Z, Xa)
            D1_ROWS.append({**base, "regime": "joint_balanced", "budget": n if n == "all" else int(n), "n_fit": int(len(fit)),
                            "tgt": resJB, **({"tgt_ext": extJB} if extJB else {}), "dentex": eval_source_v2(envJB, Xs[dte], Y[dte])})
            save_per_tooth(f"D_{b}_joint_balanced_n{n}_s{s}", arrJB)
            if extJB: save_per_tooth(f"D_{b}_joint_balanced_n{n}_s{s}_ext", extJBA)
        print(f"{b} s{s}: source-only caries full {resS['full']['auc_Caries']:.3f} | target-only(all) caries "
              f"{[r for r in D1_ROWS if r['backbone']==b and r['seed']==s and r['regime']=='target_only' and r['budget']=='all'][0]['tgt']['full']['auc_Caries']:.3f}")
save_rows(D1_ROWS, "D1_joint_budgets")
print("saved v2/D1_joint_budgets.json/.csv")

# ---------------- Part 2: CTT on dev partition ----------------
def _cluster_boot_R(ps, pt, y, groups, n_boot=N_BOOT, seed=0):
    rng = np.random.RandomState(seed); ug = np.unique(groups)
    by = defaultdict(list)
    for i, g in enumerate(groups): by[g].append(i)
    Rs = []
    for _ in range(n_boot):
        idx = np.concatenate([by[g] for g in rng.choice(ug, size=len(ug), replace=True)])
        if len(np.unique(y[idx])) < 2: continue
        at, ast = auc_safe(y[idx], pt[idx]), auc_safe(y[idx], ps[idx])
        if at - 0.5 > 0: Rs.append((at - ast) / (at - 0.5))
    return np.array(Rs)

def ctt_direction(direction, b, s, n_budget):
    """direction 'D2A': source DENTEX, target ADCD.  'A2D': source ADCD, target DENTEX."""
    Xs, Xa = load_dentex_v2(b), load_adcd_v2(b)
    dtr, dva, dte = dentex_split_v2(s); atr, ava, ate = adcd_split_v2(s)
    if direction == "D2A":
        XS, YS, s_fit, s_val = Xs, Y, dtr, dva
        XT, YT, GT, t_pool = Xa, ADCD_Y, ADCD_GROUPS, atr
    else:
        XS, YS, s_fit, s_val = Xa, ADCD_Y, atr, ava
        XT, YT, GT, t_pool = Xs, Y, DENTEX_GROUPS, dtr
    dev_fit, dev_eval = _split_idx_by_group(t_pool, GT, 0.3, s + 1000)
    envS = _fit(XS, YS, s_fit, s_val, s)
    fit_n = _budget_idx(dev_fit, n_budget, s)
    f2, v2 = _split_idx_by_group(fit_n, GT, 0.2, s) if len(fit_n) >= 32 else (fit_n, fit_n)
    envT = _fit(XT, YT, f2, v2, s)
    PS, PT = envS.proba(XT[dev_eval]), envT.proba(XT[dev_eval])
    out = []
    for c in TARGET_CLASSES:
        j = CLASS_IDX[c]; y = YT[dev_eval, j]
        ast, at = auc_safe(y, PS[:, j]), auc_safe(y, PT[:, j])
        G, H = at - ast, at - 0.5
        R = G / H if H > 0 else float("nan")
        boots = _cluster_boot_R(PS[:, j], PT[:, j], y, GT[dev_eval], seed=s) if H > 0 else np.array([])
        out.append({"direction": direction, "backbone": b, "seed": s, "budget": n_budget if n_budget == "all" else int(n_budget),
                    "n_fit": int(len(f2)), "n_dev_eval": int(len(dev_eval)), "condition": c,
                    "A_S2T": ast, "A_T": at, "G": G, "H": H, "R": R,
                    "verdict": "unlearnable" if not H > 0 else ("non-transferable" if R >= RHO else "transferable"),
                    "R_ci95": [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))] if len(boots) else None,
                    "P_R_ge_rho": float((boots >= RHO).mean()) if len(boots) else None})
    return out

D2_ROWS = []
for b in D_BACKBONES:
    for s in D_SEEDS:
        for n in CTT_BUDGETS:
            for direction in ("D2A", "A2D"):
                D2_ROWS += ctt_direction(direction, b, s, n)
    print(f"CTT done for {b}")

# outcome on untouched ADCD test (D2A only): best adapted full-coverage AUROC per condition
def _load(name):
    p = os.path.join(V2, f"{name}.json"); return json.load(open(p)) if os.path.exists(p) else []
adapted = [r for r in _load("B_frozen_ladder") + _load("C_live_trunk") if "error" not in r]
tgt_all = {(r["backbone"], r["seed"]): r for r in D1_ROWS if r["regime"] == "target_only" and r["budget"] == "all"}
tgt_matched = {(r["backbone"], r["seed"]): r for r in D1_ROWS if r["regime"] == "target_only" and r["budget"] == CTT_OUTCOME_PROTOCOL[1]}
for row in D2_ROWS:
    if row["direction"] != "D2A": continue
    b, s, c = row["backbone"], row["seed"], row["condition"]
    at_test = tgt_all[(b, s)]["tgt"]["full"][f"auc_{c}"] if (b, s) in tgt_all else float("nan")
    if not (at_test == at_test and at_test > 0.5): continue
    row["A_T_test"] = at_test
    lock = [r for r in adapted if r["backbone"] == b and r["seed"] == s and r.get("technique") == CTT_OUTCOME_PROTOCOL[0]
            and r.get("budget") == CTT_OUTCOME_PROTOCOL[1]]
    if not lock:
        print(f"WARNING: locked outcome protocol {CTT_OUTCOME_PROTOCOL} not found in Stage B results for {b} s{s}")
    if lock:
        a_p = lock[0]["tgt"]["full"][f"auc_{c}"]; r_out = (at_test - a_p) / (at_test - 0.5)
        out = "non-transferable" if r_out >= RHO else "transferable"
        row.update({"protocol": f"{CTT_OUTCOME_PROTOCOL[0]}@{CTT_OUTCOME_PROTOCOL[1]}", "A_protocol_test": a_p,
                    "R_outcome": r_out, "outcome": out, "agree": row["verdict"] == out})
        if (b, s) in tgt_matched:      # reference with the SAME target-label budget as the protocol
            at_m = tgt_matched[(b, s)]["tgt"]["full"][f"auc_{c}"]
            if at_m == at_m and at_m > 0.5:
                rm = (at_m - a_p) / (at_m - 0.5)
                row.update({"A_T_test_matched_budget": at_m, "R_outcome_matched_budget": rm,
                            "outcome_matched_budget": "non-transferable" if rm >= RHO else "transferable"})
        th = lock[0].get("controls", {}).get("tgt_head", {}).get("full", {}).get(f"auc_{c}")
        if th is not None and th == th:
            row["A_protocol_test_tgt_head"] = th; row["R_outcome_tgt_head"] = (at_test - th) / (at_test - 0.5)
    cand = [r["tgt"]["full"][f"auc_{c}"] for r in adapted if r["backbone"] == b and r["seed"] == s
            and r.get("technique", r.get("method")) not in ("source_only",)]
    cand = [x for x in cand if x == x]
    if cand:
        best = max(cand); row.update({"best_adapted_test_OPTIMISTIC": best,
                                      "R_outcome_bestof_OPTIMISTIC": (at_test - best) / (at_test - 0.5)})
save_rows(D2_ROWS, "D2_ctt")

# summary print
print("\nCTT summary (D2A, budget all, per seed):")
for b in D_BACKBONES:
    for c in TARGET_CLASSES:
        rs = [r for r in D2_ROWS if r["backbone"] == b and r["condition"] == c and r["direction"] == "D2A" and r["budget"] == "all"]
        Rv = [r["R"] for r in rs if r["R"] == r["R"]]
        ag = [r.get("agree") for r in rs if "agree" in r]
        print(f"  {b:11s} {c:18s} R per seed {[round(x, 2) for x in Rv]} | verdicts {[r['verdict'] for r in rs]} | agree with test outcome {ag}")
print("saved D2_ctt.json/.csv")

# ---------------- Part 3: crop-geometry-only baseline ----------------
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from PIL import Image as _PILImage
def _geom(x0, y0, w, h, W, H):
    return [w / W, h / H, w / max(h, 1e-6), (w * h) / (W * H), (x0 + w / 2) / W, (y0 + h / 2) / H]
_dsz = {}
def _dentex_size(f):
    if f not in _dsz:
        pth = os.path.join(IMG_DIR, os.path.basename(f))
        _dsz[f] = _PILImage.open(pth).size if os.path.exists(pth) else load_image(f).size
    return _dsz[f]
G_D = np.array([_geom(*t["bbox"], *_dentex_size(t["file"])) for t in teeth], np.float32)
G_A = np.array([_geom(*r["xywh"], r["W"], r["H"]) for r in ADCD_RECS], np.float32)
D3_ROWS = []
for s in D_SEEDS:
    dtr, dva, dte = dentex_split_v2(s); atr, ava, ate = adcd_split_v2(s)
    for c in TARGET_CLASSES:
        j = CLASS_IDX[c]
        for name, (Xtr, ytr), (Xte, yte) in (
                ("ADCD->ADCD", (G_A[atr], ADCD_Y[atr, j]), (G_A[ate], ADCD_Y[ate, j])),
                ("DENTEX->ADCD", (G_D[dtr], Y[dtr, j]), (G_A[ate], ADCD_Y[ate, j])),
                ("DENTEX->DENTEX", (G_D[dtr], Y[dtr, j]), (G_D[dte], Y[dte, j]))):
            if len(np.unique(ytr)) < 2 or len(np.unique(yte)) < 2: continue
            clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")).fit(Xtr, ytr)
            D3_ROWS.append({"seed": s, "condition": c, "setting": name, "auroc": auc_safe(yte, clf.predict_proba(Xte)[:, 1]),
                            "n_test": int(len(yte)), "pos_test": int(yte.sum())})
save_rows(D3_ROWS, "D3_geometry_baseline")
print("\nGeometry-only baseline (mean AUROC over seeds):")
for name in ("ADCD->ADCD", "DENTEX->ADCD", "DENTEX->DENTEX"):
    print("  ", name, {c: round(float(np.mean([r["auroc"] for r in D3_ROWS if r["setting"] == name and r["condition"] == c])), 3)
                       for c in TARGET_CLASSES if any(r["setting"] == name and r["condition"] == c for r in D3_ROWS)})

## 10b. Stage G: false positives on healthy DENTEX teeth (with a validity check)
Checks first that the enumeration-only and diagnosis radiographs matched by file name are the same files. If not, it reports that the healthy-tooth analysis (Fig. 7) is unsupported and stops.

In [ ]:
# =====================================================================================
# STAGE G - false positives on healthy DENTEX teeth (Fig. 7, Section V-D), with a validity check
# -------------------------------------------------------------------------------------
# The old analysis took "healthy" teeth from DENTEX's quadrant-enumeration set (enumerated teeth
# with no diagnosis) and matched them to the diagnosis set BY FILE NAME, keeping enumerated teeth
# that overlap no diagnosis box (IoU < 0.1). DENTEX documents these as separate radiograph sets
# (634 enumeration-only vs 1,005 fully labelled), so same-named files may be different radiographs.
# This stage first checks, for every file name present in both sets, whether the two image files
# are byte-identical (MD5). Only identical pairs are used. If too few are identical, the healthy-
# tooth analysis cannot be supported by public DENTEX labels and the stage says so and stops.
#
# For valid pairs: healthy teeth from DENTEX TEST-split radiographs only (per seed); a tooth counts
# as a false positive if any of the three shared conditions has probability > t. Reported at full
# coverage and on gate-accepted teeth, with image-cluster bootstrap 95% CI at t = 0.5, plus
# sensitivity on DENTEX test positives per condition at each threshold.
# =====================================================================================
import os, json, glob, hashlib, time, numpy as np
from collections import defaultdict
from PIL import Image

G_BACKBONES = [b for b in ("dental_vitl", "dinov3", "biomedclip") if b in V2_BACKBONES]
G_THRESHOLDS = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
G_MIN_IDENTICAL_FRAC = 0.9
G_IOU_MAX = 0.1

def _md5(p):
    h = hashlib.md5()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

def _iou_xywh(a, b):
    ax0, ay0, aw, ah = a; bx0, by0, bw, bh = b
    ix0, iy0, ix1, iy1 = max(ax0, bx0), max(ay0, by0), min(ax0+aw, bx0+bw), min(ay0+ah, by0+bh)
    inter = max(0, ix1-ix0) * max(0, iy1-iy0); u = aw*ah + bw*bh - inter
    return inter / u if u > 0 else 0.0

def find_enumeration_set():
    for p in glob.glob(os.path.join(DENTEX_TRAIN, "**", "*.json"), recursive=True):
        if "_results" in p: continue
        try: j = json.load(open(p))
        except Exception: continue
        if (isinstance(j, dict) and "annotations" in j and any("category_id_2" in a for a in j["annotations"])
                and not any("category_id_3" in a for a in j["annotations"])):
            root = os.path.dirname(p)
            cands = [d for d, _, fs in os.walk(root) if any(f.lower().endswith((".png", ".jpg")) for f in fs) and "_results" not in d]
            return p, j, (cands[0] if cands else None)
    return None, None, None

G_STATUS = {}
enum_path, enum, enum_img_dir = find_enumeration_set()
if enum is None or enum_img_dir is None:
    G_STATUS = {"valid": False, "reason": "enumeration-only DENTEX set not found"}
else:
    enum_files = {im["id"]: im["file_name"] for im in enum["images"]}
    diag_name2id = {v: k for k, v in img_file.items()}
    common = sorted(set(os.path.basename(f) for f in enum_files.values()) & set(os.path.basename(f) for f in img_file.values()))
    identical, different, missing = [], [], 0
    for nm in common:
        pa, pb = os.path.join(enum_img_dir, nm), os.path.join(IMG_DIR, nm)
        if not (os.path.exists(pa) and os.path.exists(pb)): missing += 1; continue
        (identical if _md5(pa) == _md5(pb) else different).append(nm)
    n_chk = len(identical) + len(different)
    frac = len(identical) / n_chk if n_chk else 0.0
    G_STATUS = {"enumeration_json": enum_path, "enumeration_images": len(enum_files), "names_in_both_sets": len(common),
                "identical_files": len(identical), "different_files": len(different), "missing_files": missing,
                "identical_fraction": frac, "valid": bool(n_chk and frac >= G_MIN_IDENTICAL_FRAC)}
    print(f"enumeration set: {len(enum_files)} images | names also in the diagnosis set: {len(common)} | "
          f"byte-identical: {len(identical)} | different radiographs: {len(different)}")
    if not G_STATUS["valid"]:
        G_STATUS["reason"] = ("same-named files in the enumeration and diagnosis sets are mostly different radiographs, "
                              "so healthy-tooth status cannot be established from public DENTEX labels; the old "
                              "healthy-tooth analysis (13,960 teeth, Fig. 7) is not supported and should be removed")
json.dump(G_STATUS, open(os.path.join(V2, "G_healthy_teeth_status.json"), "w"), indent=2)
print("G status:", {k: v for k, v in G_STATUS.items() if k != "enumeration_json"})

if G_STATUS.get("valid"):
    ident = set(identical)
    diag_boxes = defaultdict(list)
    for a in coco["annotations"]: diag_boxes[a["image_id"]].append(a["bbox"])
    HEALTHY = []
    for a in enum["annotations"]:
        nm = os.path.basename(enum_files.get(a["image_id"], ""))
        if nm not in ident: continue
        did = diag_name2id.get(nm) or diag_name2id.get(next((f for f in img_file.values() if os.path.basename(f) == nm), None))
        if did is None: continue
        if all(_iou_xywh(a["bbox"], b) < G_IOU_MAX for b in diag_boxes.get(did, [])):
            HEALTHY.append({"image_id": did, "file": img_file[did], "bbox": list(a["bbox"])})
    print(f"healthy teeth (enumerated, no overlapping diagnosis, identical radiographs): {len(HEALTHY)}")
    HG = np.array([h["image_id"] for h in HEALTHY])
    def healthy_crop(i): h = HEALTHY[i]; return crop_tooth(load_image(h["file"]), h["bbox"])
    G_ROWS = []
    for b in G_BACKBONES:
        fp = os.path.join(CACHE, f"emb_dentex_healthy_v2_{b}.npy")
        if os.path.exists(fp) and len(np.load(fp)) == len(HEALTHY): XH = np.load(fp)
        else:
            emb = make_embedder(b); XH = []; t0 = time.time()
            for k in range(0, len(HEALTHY), 256):
                XH.append(emb([healthy_crop(i) for i in range(k, min(k+256, len(HEALTHY)))]))
                print(f"  healthy/{b}: {min(k+256, len(HEALTHY))}/{len(HEALTHY)} ({time.time()-t0:.0f}s)", end="\r")
            XH = np.concatenate(XH, 0).astype(np.float32); np.save(fp, XH); print()
        Xs = load_dentex_v2(b)
        for s in SEEDS:
            dtr, dva, dte = dentex_split_v2(s)
            test_imgs = set(DENTEX_GROUPS[dte].tolist())
            hm = np.array([g in test_imgs for g in HG])
            if hm.sum() < 20: continue
            set_seed(s); env = Envelope().fit(Xs[dtr], Y[dtr], Xs[dva], Y[dva])
            PH = env.proba(XH[hm])[:, TCOLS]; accH = env.accept_mask(XH[hm]); gH = HG[hm]
            PT = env.proba(Xs[dte])[:, TCOLS]; YT = Y[dte][:, TCOLS]
            for t in G_THRESHOLDS:
                fpH = (PH > t).any(1)
                row = {"backbone": b, "seed": s, "threshold": t, "n_healthy": int(hm.sum()),
                       "fpr_full": float(fpH.mean()), "fpr_accepted": float(fpH[accH].mean()) if accH.sum() else float("nan"),
                       "healthy_coverage": float(accH.mean())}
                for j, c in enumerate(TARGET_CLASSES):
                    pos = YT[:, j] > 0
                    row[f"sens_{c}"] = float((PT[pos, j] > t).mean()) if pos.sum() else float("nan")
                if t == 0.5:   # image-cluster bootstrap CI of FPR at full coverage
                    by = defaultdict(list)
                    for i, g in enumerate(gH): by[g].append(i)
                    keys = list(by); rng = np.random.RandomState(s); bs = []
                    for _ in range(1000):
                        idx = np.concatenate([by[keys[q]] for q in rng.randint(0, len(keys), len(keys))]); bs.append(fpH[idx].mean())
                    row["fpr_full_ci"] = [float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))]
                G_ROWS.append(row)
            print(f"{b} s{s}: healthy test teeth {int(hm.sum())} | FPR@0.5 full "
                  f"{[r['fpr_full'] for r in G_ROWS if r['backbone']==b and r['seed']==s and r['threshold']==0.5][0]:.3f}")
    save_rows(G_ROWS, "G_healthy_fpr")
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(figsize=(6.5, 4))
        for b in G_BACKBONES:
            m = [np.mean([r["fpr_full"] for r in G_ROWS if r["backbone"] == b and r["threshold"] == t]) for t in G_THRESHOLDS]
            ax.plot(G_THRESHOLDS, m, marker="o", label=b)
        ax.set_xlabel("Probability threshold"); ax.set_ylabel("False-positive rate on healthy DENTEX test teeth")
        ax.grid(alpha=0.3); ax.legend(); fig.tight_layout()
        for ext in ("png", "pdf"): fig.savefig(os.path.join(V2, f"Figure_healthy_fpr_v2.{ext}"), dpi=300, facecolor="white")
        plt.close(fig)
    except Exception as e:
        print("figure skipped:", e)
    print("saved G_healthy_fpr.json/.csv")
else:
    print("Stage G skipped:", G_STATUS.get("reason"))

## 11. Stage E: statistics and manuscript tables

In [ ]:
# =====================================================================================
# STAGE E - statistics and manuscript tables from the per-tooth scores
# -------------------------------------------------------------------------------------
# Needs: section 7 and the outputs of Stages B, C, D (missing files are skipped).
#
# Reviewer 3, comment 7 asks for paired image- or patient-cluster bootstrap intervals for
# key differences and a prespecified tolerance for source preservation. This cell gives:
#   E1  per-condition AUROC, seed-mean with image-cluster bootstrap 95% CI, for every
#       run, at full coverage and under the source gate          -> v2/E1_auroc_ci.csv
#       (the old Table VI bootstrap resampled teeth, not images)
#   E2  paired differences vs source_only of the same backbone and seed, at full coverage
#       and at matched 20% / 50% coverage, image-cluster bootstrap -> v2/E2_paired_diff.csv
#   E3  source preservation: DENTEX test macro-AUROC change per run against SRC_TOL
#                                                                 -> v2/E3_source_preservation.csv
#   E4  AUROC-coverage curves for every technique (source-gate ordering)
#                                                                 -> v2/E4_auroc_coverage.csv + figure
#   E5  manuscript-ready summary tables (seed means, coverage, accepted counts, AUPRC, controls)
#   E6  refusal matrix: source/target gate x DENTEX/ADCD/ADCD-external     -> v2/E6_gate_matrix.csv
#                                                                 -> v2/E5_table_*.csv
#
# =====================================================================================
import numpy as np, os, json, glob, csv
from collections import defaultdict

# SRC_TOL is set in section 2
N_BOOT_E = 1000
CLASSES3 = list(TARGET_CLASSES)

def _load_rows(name):
    p = os.path.join(V2, f"{name}.json"); return [r for r in json.load(open(p)) if "error" not in r] if os.path.exists(p) else []
B_ = _load_rows("B_frozen_ladder"); C_ = _load_rows("C_live_trunk"); D1_ = _load_rows("D1_joint_budgets"); F_ = _load_rows("B2_fusion")
for r in B_: r["name"] = r["technique"]
for r in C_: r["name"] = r["method"]
for r in F_: r["name"] = r["technique"]
for r in D1_: r["name"] = f"{r['regime']}"; r["per_tooth"] = f"D_{r['backbone']}_{r['regime']}_n{r['budget']}_s{r['seed']}"
ALL = B_ + C_ + D1_ + F_
def _key(r): return (r.get("cell", "D"), r["backbone"], r["name"], str(r.get("budget", 0)))

def _pt(tag): return dict(np.load(os.path.join(PT_DIR, f"{tag}.npz"), allow_pickle=False))

def _masks(a):
    n = len(a["labels"]); out = {"full": np.ones(n, bool), "src": a["maha_src"] <= a["tau_src"]}
    order = np.argsort(a["maha_src"])
    for f in MATCHED_COVERAGE:
        m = np.zeros(n, bool); m[order[:max(1, int(round(f*n)))]] = True; out[f"cov{int(f*100)}"] = m
    if "maha_tgt" in a: out["tgt"] = a["maha_tgt"] <= a["tau_tgt"]
    if "maha_fixed" in a:       # one tooth set shared by all techniques (unadapted source gate ordering)
        fo = np.argsort(a["maha_fixed"])
        for f in MATCHED_COVERAGE:
            m = np.zeros(n, bool); m[fo[:max(1, int(round(f*n)))]] = True; out[f"fix{int(f*100)}"] = m
    return out

def _group_index(groups):
    by = defaultdict(list)
    for i, g in enumerate(groups): by[g].append(i)
    return {g: np.array(v) for g, v in by.items()}

from scipy.stats import rankdata
def _fast_auc(y, p):
    y = np.asarray(y).astype(bool); n1 = y.sum(); n0 = len(y) - n1
    if n1 == 0 or n0 == 0: return float("nan")
    r = rankdata(p); return float((r[y].sum() - n1*(n1+1)/2) / (n1*n0))

MIN_POS = MIN_NEG = 5      # AUROC left blank unless a condition has at least 5 positives and 5 negatives
def _auc_on(a, mask, j, idx=None):
    sel = np.where(mask)[0] if idx is None else idx[mask[idx]]
    y = a["labels"][sel, j]; npos = int(y.sum())
    if npos < MIN_POS or len(y) - npos < MIN_NEG: return float("nan")
    return _fast_auc(y, a["probs"][sel, j])

def _macro_strict(vals):
    """Macro over the three conditions only when all three are evaluable; otherwise NaN."""
    vals = list(vals); return float(np.mean(vals)) if all(v == v for v in vals) else float("nan")

# ---------------- E1: AUROC with image-cluster bootstrap (seed mean) ----------------
groups_of_runs = defaultdict(list)
for r in ALL: groups_of_runs[_key(r)].append(r)
E1 = []
rng = np.random.RandomState(0)
for key, runs in groups_of_runs.items():
  for suffix in ("", "_ext"):
    if suffix and not all(os.path.exists(os.path.join(PT_DIR, r["per_tooth"] + suffix + ".npz")) for r in runs): continue
    arrs = [_pt(r["per_tooth"] + suffix) for r in runs]
    gidx = [_group_index(a["groups"]) for a in arrs]
    for view in ("full", "src", "tgt", "cov20", "cov50", "fix20", "fix50"):
        if view == "tgt" and not all("maha_tgt" in a for a in arrs): continue
        if view.startswith("fix") and not all("maha_fixed" in a for a in arrs): continue
        ms = [_masks(a)[view] for a in arrs]
        points = [np.nanmean([_auc_on(a, m, j) for a, m in zip(arrs, ms)]) for j in range(3)]
        boots = [[] for _ in range(3)]
        for _ in range(N_BOOT_E):
            idxs = []
            for gi in gidx:
                keys = list(gi.keys()); pick = rng.randint(0, len(keys), len(keys))
                idxs.append(np.concatenate([gi[keys[p]] for p in pick]))
            for j in range(3):
                vals = [_auc_on(a, m, j, idx) for a, m, idx in zip(arrs, ms, idxs)]
                if not np.all(np.isnan(vals)): boots[j].append(np.nanmean(vals))
        for j, c in enumerate(CLASSES3):
            lo, hi = (np.percentile(boots[j], [2.5, 97.5]) if boots[j] else (np.nan, np.nan))
            E1.append({"cell": key[0], "backbone": key[1], "technique": key[2], "budget": key[3], "view": ("ext_" if suffix else "") + view,
                       "condition": c, "auroc_seed_mean": points[j], "ci_lo": lo, "ci_hi": hi, "n_seeds": len(arrs),
                       "above_chance": bool(lo > 0.5), "coverage_mean": float(np.mean([m.mean() for m in ms])),
                       "accepted_pos_mean": float(np.mean([a["labels"][m, j].sum() for a, m in zip(arrs, ms)]))})
    print("E1", key)
with open(os.path.join(V2, "E1_auroc_ci.csv"), "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(E1[0].keys())); w.writeheader(); w.writerows(E1)

# ---------------- E2: paired differences vs source_only ----------------
ref = {}
for r in B_:
    if r["name"] == "source_only": ref[(r["backbone"], r["seed"])] = r["per_tooth"]
E2 = []
for key, runs in groups_of_runs.items():
    if key[2] == "source_only" or key[0] == "D": continue
    pairs = [(r, ref.get((r["backbone"], r["seed"]))) for r in runs]
    pairs = [(r, t) for r, t in pairs if t]
    if not pairs: continue
    A = [_pt(r["per_tooth"]) for r, _ in pairs]; R0 = [_pt(t) for _, t in pairs]
    for a, b0 in zip(A, R0):
        assert np.array_equal(a["groups"], b0["groups"]), "paired runs must share the ADCD test split"
    gidx = [_group_index(a["groups"]) for a in A]
    for view in ("full", "cov20", "cov50", "fix20", "fix50"):
        if view.startswith("fix") and not all("maha_fixed" in a for a in A + R0): continue
        ma = [_masks(a)[view] for a in A]; mb = [_masks(b0)[view] for b0 in R0]
        for j, c in enumerate(CLASSES3 + ["macro"]):
            def diff(idx_list=None):
                vals = []
                for k_, (a, b0, m1, m2) in enumerate(zip(A, R0, ma, mb)):
                    idx = None if idx_list is None else idx_list[k_]
                    if c == "macro":
                        d = _macro_strict(_auc_on(a, m1, jj, idx) for jj in range(3)) - _macro_strict(_auc_on(b0, m2, jj, idx) for jj in range(3))
                    else:
                        d = _auc_on(a, m1, j, idx) - _auc_on(b0, m2, j, idx)
                    vals.append(d)
                return np.nanmean(vals)
            point = diff(); boots = []
            for _ in range(N_BOOT_E):
                idxs = []
                for gi in gidx:
                    keys = list(gi.keys()); pick = rng.randint(0, len(keys), len(keys))
                    idxs.append(np.concatenate([gi[keys[p]] for p in pick]))
                v = diff(idxs)
                if v == v: boots.append(v)
            lo, hi = np.percentile(boots, [2.5, 97.5]) if boots else (np.nan, np.nan)
            E2.append({"cell": key[0], "backbone": key[1], "technique": key[2], "budget": key[3], "view": view,
                       "condition": c, "diff_vs_source_only": point, "ci_lo": lo, "ci_hi": hi,
                       "excludes_zero": bool(lo > 0 or hi < 0), "n_seeds": len(A)})
    print("E2", key)
if E2:
    with open(os.path.join(V2, "E2_paired_diff.csv"), "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(E2[0].keys())); w.writeheader(); w.writerows(E2)

# ---------------- E3: source preservation against SRC_TOL ----------------
E3 = []
for r in B_ + C_:
    drop = r["src_before"]["macro4_full"] - r["src_after"]["macro4_full"]
    row3 = {"cell": r["cell"], "backbone": r["backbone"], "technique": r["name"], "budget": r.get("budget"), "seed": r["seed"],
            "dentex_before": r["src_before"]["macro4_full"], "dentex_after": r["src_after"]["macro4_full"],
            "drop": drop, "within_tol": bool(drop <= SRC_TOL),
            "dentex3_before": r["src_before"]["macro3_full"], "dentex3_after": r["src_after"]["macro3_full"],
            "drop3": r["src_before"]["macro3_full"] - r["src_after"]["macro3_full"], "delta_ok": r["delta_ok"]}
    for c in CLASSES:
        if f"auc_{c}" in r["src_before"] and f"auc_{c}" in r["src_after"]:
            row3[f"drop_{c}"] = r["src_before"][f"auc_{c}"] - r["src_after"][f"auc_{c}"]
    E3.append(row3)
with open(os.path.join(V2, "E3_source_preservation.csv"), "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(E3[0].keys())); w.writeheader(); w.writerows(E3)
print(f"E3: {sum(x['within_tol'] for x in E3)}/{len(E3)} runs within SRC_TOL={SRC_TOL}; "
      f"largest drop {max(x['drop'] for x in E3):.3f}; delta violations {sum(not x['delta_ok'] for x in E3)}")

# ---------------- E4: AUROC-coverage curves ----------------
E4 = []; fracs = np.linspace(0.1, 1.0, 19)
for key, runs in groups_of_runs.items():
    for f in fracs:
        vals = []
        for r in runs:
            a = _pt(r["per_tooth"]); order = np.argsort(a["maha_src"]); m = np.zeros(len(order), bool)
            m[order[:max(1, int(round(f*len(order))))]] = True
            vals.append(_macro_strict(_auc_on(a, m, j) for j in range(3)))
        E4.append({"cell": key[0], "backbone": key[1], "technique": key[2], "budget": key[3],
                   "coverage": round(float(f), 3), "n_seeds_defined": int(sum(v == v for v in vals)),
                   "macro_mean": float(np.mean(vals)) if all(v == v for v in vals) else float("nan"),
                   "macro_sd": float(np.std(vals)) if all(v == v for v in vals) else float("nan")})
with open(os.path.join(V2, "E4_auroc_coverage.csv"), "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(E4[0].keys())); w.writeheader(); w.writerows(E4)
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for key in groups_of_runs:
        if key[1] != "dental_vitl" or key[0] == "D" or key[3] not in ("0", "256"): continue
        pts = [e for e in E4 if (e["cell"], e["backbone"], e["technique"], e["budget"]) == key]
        ax.plot([p["coverage"] for p in pts], [p["macro_mean"] for p in pts], label=f"{key[2]} ({key[0]})", lw=1.2)
    ax.axhline(0.5, ls="--", color="grey"); ax.set_xlabel("Coverage (fraction of ADCD test crops scored)")
    ax.set_ylabel("Macro-AUROC over three conditions"); ax.legend(fontsize=6, ncol=2, loc="lower left")
    fig.tight_layout()
    for ext in ("png", "pdf"): fig.savefig(os.path.join(V2, f"Figure_auroc_coverage_v2.{ext}"), dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
except Exception as e:
    print("figure skipped:", e)

# ---------------- E5: manuscript summary tables ----------------
def _m(v): v = [x for x in v if x == x]; return float(np.mean(v)) if v else float("nan")
def _sd(v): v = [x for x in v if x == x]; return float(np.std(v, ddof=1)) if len(v) > 1 else float("nan")
for cellname, rows in (("B", B_), ("C", C_), ("D1", D1_), ("F", F_)):
    if not rows: continue
    grp = defaultdict(list)
    for r in rows: grp[(r["backbone"], r["name"], str(r.get("budget", 0)))].append(r)
    out = []
    for (b, t, n), rs in grp.items():
        row = {"backbone": b, "technique": t, "budget": n, "n_seeds": len(rs),
               "params": rs[0].get("trainable", rs[0].get("adapt_params", "")), "n_test_mean": _m([r["tgt"]["full"]["n"] for r in rs])}
        for key_t, pre in (("tgt", ""), ("tgt_ext", "ext_")):
          if key_t not in rs[0]: continue
          for view in ("full", "src", "tgt", "cov20", "cov50", "fix20", "fix50"):
            if view not in rs[0][key_t]: continue
            V = pre + view
            ok_runs = [r for r in rs if all(r[key_t][view].get(f"pos_{c}", 0) >= MIN_POS and r[key_t][view].get(f"neg_{c}", MIN_NEG) >= MIN_NEG for c in CLASSES3)]
            row[f"{V}_runs_meeting_min_counts"] = len(ok_runs)
            row[f"{V}_macro"] = _m([r[key_t][view]["macro"] for r in ok_runs]) if len(ok_runs) == len(rs) else float("nan")
            row[f"{V}_macro_sd"] = _sd([r[key_t][view]["macro"] for r in ok_runs]) if len(ok_runs) == len(rs) else float("nan")
            row[f"{V}_coverage"] = _m([r[key_t][view]["coverage"] for r in rs])
            for c in CLASSES3:
                row[f"{V}_{c}"] = _m([r[key_t][view][f"auc_{c}"] for r in rs])
                row[f"{V}_pos_{c}"] = _m([r[key_t][view][f"pos_{c}"] for r in rs])
            row[f"{V}_ece"] = _m([r[key_t][view]["ece"] for r in rs])
            for c in CLASSES3:
                row[f"{V}_auprc_{c}"] = _m([r[key_t][view].get(f"auprc_{c}", float("nan")) for r in rs])
                row[f"{V}_prev_{c}"] = _m([r[key_t][view].get(f"prev_{c}", float("nan")) for r in rs])
        for ctrl in ("tgt_head", "src_env_fixed"):          # control heads on the same test crops
            if "controls" not in rs[0] or ctrl not in rs[0]["controls"]: continue
            for view in ("full", "src"):
                row[f"ctrl_{ctrl}_{view}_macro"] = _m([r["controls"][ctrl][view]["macro"] for r in rs])
                for c in CLASSES3:
                    row[f"ctrl_{ctrl}_{view}_{c}"] = _m([r["controls"][ctrl][view][f"auc_{c}"] for r in rs])
        # parameter accounting: adaptation, temporary readout/head, and envelope, reported separately
        d_ = {"dental_vitl": 1024, "dinov3": 1024, "biomedclip": 512}.get(b, float("nan"))
        row["params_adaptation"] = rs[0].get("adapt_params", rs[0].get("trainable", ""))
        row["params_readout"] = rs[0].get("readout_params", (d_ * len(CLASSES3)) if str(t).startswith("E_") else 0)
        row["params_envelope"] = d_ * NUM_DX + NUM_DX + 1 + (d_ * 256 + 256 + 256 * NUM_DX + NUM_DX)
        if "src_after" in rs[0]:
            row["dentex_before"] = _m([r["src_before"]["macro4_full"] for r in rs]); row["dentex_after"] = _m([r["src_after"]["macro4_full"] for r in rs])
            row["delta_ok_all"] = all(r["delta_ok"] for r in rs)
        if "dentex" in rs[0]: row["dentex_macro4"] = _m([r["dentex"]["macro4_full"] for r in rs])
        if "train_minutes" in rs[0]: row["train_minutes_mean"] = _m([r["train_minutes"] for r in rs])
        out.append(row)
    keys = sorted({k for r in out for k in r}, key=lambda k: (k not in ("backbone", "technique", "budget", "n_seeds", "params"), k))
    with open(os.path.join(V2, f"E5_table_{cellname}.csv"), "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=keys); w.writeheader(); w.writerows(out)
# ---------------- E6: refusal matrix, gate fitted on source or target x evaluated on source or target ----------------
E6 = []
grp6 = defaultdict(list)
for r in B_ + C_: grp6[(r["cell"], r["backbone"], r["name"], str(r.get("budget", 0)))].append(r)
for (cell_, b, t, n), rs in grp6.items():
    E6.append({"cell": cell_, "backbone": b, "technique": t, "budget": n, "n_seeds": len(rs),
               "srcgate_on_DENTEX": _m([r["src_after"]["refusal"] for r in rs]),
               "srcgate_on_ADCD": _m([1 - r["tgt"]["src"]["coverage"] for r in rs]),
               "tgtgate_on_ADCD": _m([1 - r["tgt"]["tgt"]["coverage"] for r in rs if "tgt" in r["tgt"]]),
               "tgtgate_on_DENTEX": _m([r["src_after"].get("tgt_gate_refusal", float("nan")) for r in rs]),
               "srcgate_on_ADCDext": _m([1 - r["tgt_ext"]["src"]["coverage"] for r in rs if "tgt_ext" in r]),
               "tgtgate_on_ADCDext": _m([1 - r["tgt_ext"]["tgt"]["coverage"] for r in rs if "tgt_ext" in r and "tgt" in r["tgt_ext"]])})
if E6:
    with open(os.path.join(V2, "E6_gate_matrix.csv"), "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=list(E6[0].keys())); w.writeheader(); w.writerows(E6)
print("saved E1-E6 in", V2)
print("Summary CSV/JSON files are in the results folder; per_tooth/ holds the per-crop scores.")

## 12. Completeness check
Checks every expected configuration, prediction file and safeguard, and records the verdict. The package cell then writes the zip (named COMPLETE or INCOMPLETE) and stops with an error if anything is missing.

In [ ]:
# =====================================================================================
# COMPLETENESS CHECK - every expected backbone x method x seed must be present without error
# (a notebook that finished is not the same as an experiment that completed). Writes completeness.json
# and stops with an error if anything is missing, so incomplete results are never mistaken for final ones.
# =====================================================================================
def _rows(name):
    p = os.path.join(V2, f"{name}.json"); return json.load(open(p)) if os.path.exists(p) else []
REPORT = {}
REPORT["required_backbones_missing"] = sorted(set(("dental_vitl", "biomedclip", "dinov3")) - set(V2_BACKBONES))
REPORT["cache_checks"] = json.load(open(os.path.join(V2, "checks.json"))) if os.path.exists(os.path.join(V2, "checks.json")) else "missing"

LABEL_FREE = {"source_only", "R0_thresh", "R0_moment"}
Brows = _rows("B_frozen_ladder")
gotB = {(r["backbone"], r["technique"], r["seed"], r["budget"]) for r in Brows if "error" not in r}
expB = {(b, t, s_, 0 if t in LABEL_FREE else n) for (b, t, s_, n) in plan_B()}
REPORT["B"] = {"expected": len(expB), "missing": sorted(map(str, expB - gotB)), "errors": [r for r in Brows if "error" in r]}
Crows = _rows("C_live_trunk")
C_ALL = ["lora", "adapter", "vpt", "bitfit", "ln_affine", "full_ewc", "full_rehearsal", "full_featanchor"]
expC = {(b, m, s_) for b in ("dental_vitl", "dinov3") if b in V2_BACKBONES for m in C_ALL for s_ in (42, 43, 44)}
gotC = {(r["backbone"], r["method"], r["seed"]) for r in Crows if "error" not in r}
REPORT["C"] = {"expected": len(expC), "missing": sorted(map(str, expC - gotC)), "errors": [r for r in Crows if "error" in r],
               "full_trunk_without_source_warm_start": sorted(str((r["backbone"], r["method"], r["seed"])) for r in Crows
                    if r.get("method", "").startswith("full_") and "error" not in r and not r.get("source_head_fit_loss_per_epoch"))}
Frows = _rows("B2_fusion")
REPORT["B2"] = {"expected": 4 * len(SEEDS) if {"dental_vitl", "dinov3"} <= set(V2_BACKBONES) else 0, "found": len(Frows),
                "without_external": sum("tgt_ext" not in r for r in Frows)}
D1 = _rows("D1_joint_budgets")
REPORT["D1"] = {"expected": len(V2_BACKBONES) * len(SEEDS) * (1 + 3 * len(JOINT_BUDGETS)), "found": len(D1),
                "balanced_joint_rows": sum(r.get("regime") == "joint_balanced" for r in D1)}
D2 = _rows("D2_ctt")
REPORT["D2"] = {"expected": len(V2_BACKBONES) * len(SEEDS) * len(CTT_BUDGETS) * 2 * 3, "found": len(D2),
                "with_locked_outcome": sum("outcome" in r for r in D2), "with_matched_budget_outcome": sum("outcome_matched_budget" in r for r in D2)}
REPORT["G"] = json.load(open(os.path.join(V2, "G_healthy_teeth_status.json"))) if os.path.exists(os.path.join(V2, "G_healthy_teeth_status.json")) else "missing"
REPORT["E_files"] = {f: os.path.exists(os.path.join(V2, f)) for f in
                     ("E1_auroc_ci.csv", "E2_paired_diff.csv", "E3_source_preservation.csv", "E4_auroc_coverage.csv", "E5_table_B.csv", "E5_table_C.csv", "E6_gate_matrix.csv")}
REPORT["delta_violations"] = [str((r["backbone"], r.get("technique", r.get("method")), r["seed"])) for r in Brows + Crows if "error" not in r and not r.get("delta_ok", True)]
REPORT["prediction_files_missing"] = [r.get("per_tooth", "<missing tag>") for r in Brows + Crows + Frows
    if "error" not in r and (not r.get("per_tooth") or not os.path.isfile(os.path.join(PT_DIR, r["per_tooth"] + ".npz")))]
REPORT["external_prediction_files_missing"] = [r.get("per_tooth", "<missing tag>") for r in Brows + Crows + Frows
    if "error" not in r and HAS_EXT and ("tgt_ext" not in r or not os.path.isfile(os.path.join(PT_DIR, r.get("per_tooth", "") + "_ext.npz")))]
REPORT["source_preservation_failures"] = [str((r["backbone"], r.get("technique", r.get("method")), r["seed"]))
    for r in Brows + Crows if "error" not in r and r.get("src_before", {}).get("macro4_full", float("nan")) - r.get("src_after", {}).get("macro4_full", float("nan")) > SRC_TOL]
REPORT["C_wrong_signature"] = [str((r["backbone"], r["method"], r["seed"])) for r in Crows if "error" not in r and r.get("run_signature") != C_RUN_SIGNATURE]

json.dump(REPORT, open(os.path.join(V2, "completeness.json"), "w"), indent=2, default=str)
problems = []
if REPORT["required_backbones_missing"]: problems.append("Required backbones unavailable: " + ", ".join(REPORT["required_backbones_missing"]))
if REPORT["cache_checks"] == "missing": problems.append("Cache and duplicate-image checks not executed")
if REPORT["prediction_files_missing"]: problems.append(f"Missing {len(REPORT['prediction_files_missing'])} prediction NPZ files")
if REPORT["external_prediction_files_missing"]: problems.append(f"Missing {len(REPORT['external_prediction_files_missing'])} external prediction NPZ files")
if REPORT["C_wrong_signature"]: problems.append(f"Stage C: {len(REPORT['C_wrong_signature'])} runs were generated under another protocol")
if REPORT["delta_violations"]: problems.append(f"Bounded-deviation violations: {len(REPORT['delta_violations'])}")

if REPORT["B"]["missing"] or REPORT["B"]["errors"]: problems.append(f"Stage B: {len(REPORT['B']['missing'])} missing, {len(REPORT['B']['errors'])} errors")
if REPORT["C"]["missing"] or REPORT["C"]["errors"]: problems.append(f"Stage C: {len(REPORT['C']['missing'])} missing, {len(REPORT['C']['errors'])} errors")
if REPORT["C"]["full_trunk_without_source_warm_start"]: problems.append(f"Stage C: {len(REPORT['C']['full_trunk_without_source_warm_start'])} full-trunk runs lack the common source warm start")
if REPORT["B2"]["found"] < REPORT["B2"]["expected"] or REPORT["B2"]["without_external"]: problems.append("Stage B2 incomplete or missing external results")
if REPORT["D1"]["found"] < REPORT["D1"]["expected"]: problems.append(f"Stage D1: {REPORT['D1']['found']}/{REPORT['D1']['expected']}")
if REPORT["D2"]["found"] < REPORT["D2"]["expected"]: problems.append(f"Stage D2: {REPORT['D2']['found']}/{REPORT['D2']['expected']}")
if not all(REPORT["E_files"].values()): problems.append("Stage E files missing")
print(json.dumps({k: (v if not isinstance(v, dict) else {kk: (vv if not isinstance(vv, list) else len(vv)) for kk, vv in v.items()}) for k, v in REPORT.items()}, indent=1, default=str))
if REPORT["source_preservation_failures"]:
    print("SAFETY RESULT: source-performance drops exceed SRC_TOL in", len(REPORT["source_preservation_failures"]),
          "configurations. Do not claim source preservation for those runs.")
COMPLETENESS_PROBLEMS = list(problems)
json.dump({"complete": not problems, "problems": problems}, open(os.path.join(V2, "completeness_verdict.json"), "w"), indent=2)
if problems:
    print("EXPERIMENT INCOMPLETE:\n  " + "\n  ".join(problems))
    print("The package cell below still writes a zip, named INCOMPLETE, and then stops with an error.")
else:
    print("COMPLETE: all expected configurations present, no errors.")

## 13. Package results
Always writes the archive. If the experiment is incomplete the file name says INCOMPLETE and this cell ends with an error.


In [ ]:
import shutil
_status = "COMPLETE" if not COMPLETENESS_PROBLEMS else "INCOMPLETE"
v2 = os.path.join(OUT2, V2_NAME); pack = os.path.join(OUT2, f"{V2_NAME}_{_status}_results")
for _old in glob.glob(os.path.join(OUT2, f"{V2_NAME}_*_results*")):
    shutil.rmtree(_old) if os.path.isdir(_old) else os.remove(_old)
shutil.copytree(v2, pack, ignore=shutil.ignore_patterns("checkpoints"))   # includes per_tooth/ scores
zipf = shutil.make_archive(pack, "zip", pack)
print("results archive:", zipf, f"({os.path.getsize(zipf)/1e6:.1f} MB)")
if COMPLETENESS_PROBLEMS:
    raise AssertionError("EXPERIMENT INCOMPLETE (zip written and named INCOMPLETE):\n  " + "\n  ".join(COMPLETENESS_PROBLEMS)
                         + "\nSee completeness.json. Do not use these results as final.")
print("Results are complete.")